# qfa_roe 因子

qfa_roe=单季度净资产收益率

## 裸因子指标计算

In [ ]:
# -*- coding: utf-8 -*-
"""
BigQuant / DAI 复现华泰财务质量因子 qfa_roe：裸因子指标测试版 v1。

口径说明：
1. 研报 qfa_roe = 单季度 ROE（平均），即单季度净利润 / 期初期末平均净资产。
2. BigQuant 官网数据字段中，qfa_roe 对应口径优先使用：roe_avg_mrq
   字段含义：净资产收益率(平均)(单季度)。
3. 本代码只使用 dai.query，不调用旧版 D.features / D.financial_statements。
4. 裸因子版本：不做市值中性化、不做行业中性化；只做截面 MAD 去极值与标准化。
5. 每隔 rebalance_freq 个交易日取一个截面，计算未来 forward_days 个交易日收益的
   IC、RankIC、WLS 因子收益率和 t 值。
6. 股票池剔除风险警示/ST、当前停牌、下一交易日停牌、北交所；收益使用日收盘价。
7. 回归法：未来 forward_days 日相对沪深300超额收益 ~ const + qfa_roe 裸标准化因子，
   WLS 权重为 sqrt(流通市值)。该权重只用于回归估计稳定性，不代表做了市值中性化。
8. 官方字段优先读取 cn_stock_prefactors.roe_avg_mrq；若该字段不可用，尝试
   cn_stock_factors_financial_indicators.roe_avg_mrq。

如果报字段不存在，请先运行：
    test_factor_sources(CFG)
检查当前 BigQuant / DAI 环境中 qfa_roe 等价字段是否可读。
"""

import os
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import display

try:
    import dai
except ImportError:
    from bigquant import dai  # type: ignore

warnings.filterwarnings("ignore")


@dataclass(frozen=True)
class FactorSource:
    table: str
    field: str
    desc: str


@dataclass(frozen=True)
class Config:
    start_date: str = "2020-01-01"
    end_date: str = "2026-06-30"

    forward_days: int = 30
    rebalance_freq: int = 30
    min_cross_section_size: int = 30

    factor_name: str = "qfa_roe_raw"
    chinese_font_path: str = ""

    # qfa_roe 的 BigQuant 等价字段：roe_avg_mrq = 净资产收益率(平均)(单季度)
    # 不回退到 roe_avg_lf / roe_avg_ttm，避免改变因子口径。
    factor_sources: Tuple[FactorSource, ...] = field(default_factory=lambda: (
        FactorSource("cn_stock_prefactors", "roe_avg_mrq", "净资产收益率(平均)(单季度)，对应研报 qfa_roe"),
        FactorSource("cn_stock_factors_financial_indicators", "roe_avg_mrq", "净资产收益率(平均)(单季度)，对应研报 qfa_roe"),
    ))


CFG = Config()


def test_factor_sources(cfg: Config = CFG) -> pd.DataFrame:
    """逐个测试 qfa_roe 等价字段是否可在当前 BigQuant / DAI 环境中读取。"""
    rows: List[Dict[str, object]] = []
    for src in cfg.factor_sources:
        sql = f"""
        SELECT date, instrument, {src.field} AS factor_raw
        FROM {src.table}
        WHERE date >= DATE '{cfg.start_date}'
          AND date <= DATE '{cfg.end_date}'
          AND {src.field} IS NOT NULL
        LIMIT 5
        """
        try:
            tmp = dai.query(sql, filters={"date": [cfg.start_date, cfg.end_date]}).df()
            ok = not tmp.empty
            rows.append({
                "table": src.table,
                "field": src.field,
                "desc": src.desc,
                "available": ok,
                "rows": len(tmp),
                "error": "" if ok else "查询成功但无非空样本",
            })
        except Exception as e:
            rows.append({
                "table": src.table,
                "field": src.field,
                "desc": src.desc,
                "available": False,
                "rows": 0,
                "error": str(e).split("\n")[-1][:240],
            })
    out = pd.DataFrame(rows)
    display(out)
    return out


def _font_has_chinese(font_path: str) -> bool:
    try:
        ft = font_manager.get_font(font_path)
        cmap = ft.get_charmap()
        return all(ord(ch) in cmap for ch in "因子日期相关系数")
    except Exception:
        return False


def _candidate_font_paths(user_font_path: str = "") -> List[str]:
    candidates: List[str] = []
    if user_font_path:
        candidates.append(user_font_path)
    env_font = os.environ.get("CHINESE_FONT_PATH", "")
    if env_font:
        candidates.append(env_font)
    candidates.extend([
        "./SimHei.ttf", "./simhei.ttf", "./msyh.ttc", "./Microsoft YaHei.ttf",
        "./NotoSansCJK-Regular.ttc", "/home/jovyan/work/SimHei.ttf",
        "/home/jovyan/work/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/truetype/wqy/wqy-microhei.ttc",
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/opentype/noto/NotoSansCJKsc-Regular.otf",
        "/System/Library/Fonts/PingFang.ttc",
        "C:/Windows/Fonts/msyh.ttc", "C:/Windows/Fonts/simhei.ttf",
    ])

    search_roots = [Path.cwd(), Path.home(), Path("/usr/share/fonts"), Path("/usr/local/share/fonts")]
    name_keywords = (
        "NotoSansCJK", "NotoSansSC", "SourceHanSans", "WenQuanYi", "wqy",
        "SimHei", "simhei", "msyh", "PingFang", "Arial Unicode",
    )
    suffixes = {".ttf", ".ttc", ".otf"}
    for root in search_roots:
        if not root.exists():
            continue
        try:
            for p in root.rglob("*"):
                if p.suffix.lower() in suffixes and any(k.lower() in p.name.lower() for k in name_keywords):
                    candidates.append(str(p))
        except Exception:
            continue

    seen = set()
    unique = []
    for p in candidates:
        pp = str(Path(p).expanduser())
        if pp not in seen:
            seen.add(pp)
            unique.append(pp)
    return unique


def set_chinese_font(font_path: str = "") -> Optional[font_manager.FontProperties]:
    preferred_names = [
        "Microsoft YaHei", "SimHei", "Noto Sans CJK SC", "Noto Sans SC",
        "Source Han Sans SC", "WenQuanYi Micro Hei", "PingFang SC", "Arial Unicode MS",
    ]
    plt.rcParams["axes.unicode_minus"] = False
    for path in _candidate_font_paths(font_path):
        if not Path(path).exists() or not _font_has_chinese(path):
            continue
        try:
            font_manager.fontManager.addfont(path)
            prop = font_manager.FontProperties(fname=path)
            font_name = prop.get_name()
            plt.rcParams["font.family"] = "sans-serif"
            plt.rcParams["font.sans-serif"] = [font_name] + preferred_names + ["DejaVu Sans"]
            return prop
        except Exception:
            continue

    available = {f.name for f in font_manager.fontManager.ttflist}
    for name in preferred_names:
        if name in available:
            plt.rcParams["font.family"] = "sans-serif"
            plt.rcParams["font.sans-serif"] = [name] + ["DejaVu Sans"]
            return font_manager.FontProperties(family=name)

    plt.rcParams["font.family"] = "sans-serif"
    plt.rcParams["font.sans-serif"] = preferred_names + ["DejaVu Sans"]
    return None


def _query_panel_with_source(cfg: Config, src: FactorSource, fetch_end: str) -> pd.DataFrame:
    """使用指定候选表和字段读取完整测试面板。"""
    if src.table == "cn_stock_prefactors":
        factor_join = ""
        factor_select = "b.factor_from_base AS factor_raw"
        factor_filter = "b.factor_from_base IS NOT NULL"
    else:
        factor_join = f"""
        JOIN {src.table} AS f
          ON b.date = f.date AND b.instrument = f.instrument
        """
        factor_select = f"f.{src.field} AS factor_raw"
        factor_filter = f"f.{src.field} IS NOT NULL"

    sql = f"""
    PRAGMA enable_pushdown_window;

    WITH trading_dates AS (
        SELECT
            date,
            ROW_NUMBER() OVER (ORDER BY date) AS rn
        FROM (
            SELECT DISTINCT date
            FROM cn_stock_prefactors
            WHERE date >= DATE '{cfg.start_date}'
              AND date <= DATE '{cfg.end_date}'
        )
    ),

    signal_dates AS (
        SELECT date
        FROM trading_dates
        WHERE MOD(rn - 1, {cfg.rebalance_freq}) = 0
    ),

    base AS (
        SELECT
            date,
            instrument,
            close,
            float_market_cap,
            is_risk_warning,
            suspended,
            list_sector,
            {src.field if src.table == "cn_stock_prefactors" else "NULL"} AS factor_from_base,
            LEAD(close, {cfg.forward_days}) OVER (
                PARTITION BY instrument ORDER BY date
            ) AS close_fwd,
            LEAD(suspended, 1) OVER (
                PARTITION BY instrument ORDER BY date
            ) AS next_suspended
        FROM cn_stock_prefactors
        WHERE date >= DATE '{cfg.start_date}'
          AND date <= DATE '{fetch_end}'
          AND COALESCE(list_sector, 0) != 4
    ),

    bench_raw AS (
        SELECT
            date,
            MAX(close_000300SH) AS hs300_close
        FROM cn_stock_prefactors
        WHERE date >= DATE '{cfg.start_date}'
          AND date <= DATE '{fetch_end}'
        GROUP BY date
    ),

    bench AS (
        SELECT
            date,
            hs300_close,
            LEAD(hs300_close, {cfg.forward_days}) OVER (ORDER BY date) AS hs300_close_fwd
        FROM bench_raw
    )

    SELECT
        b.date,
        b.instrument,
        b.float_market_cap,
        {factor_select},
        b.close_fwd / b.close - 1.0 AS ret_fwd,
        b.close_fwd / b.close - 1.0
            - (be.hs300_close_fwd / be.hs300_close - 1.0) AS excess_ret_fwd
    FROM base AS b
    JOIN signal_dates AS sd
      ON b.date = sd.date
    {factor_join}
    JOIN bench AS be
      ON b.date = be.date
    WHERE b.date <= DATE '{cfg.end_date}'
      AND b.is_risk_warning = 0
      AND b.suspended = 0
      AND COALESCE(b.next_suspended, 1) = 0
      AND b.close > 0
      AND b.close_fwd > 0
      AND be.hs300_close > 0
      AND be.hs300_close_fwd > 0
      AND b.float_market_cap > 0
      AND {factor_filter}
    ORDER BY b.date, b.instrument
    """
    return dai.query(sql, filters={"date": [cfg.start_date, fetch_end]}).df()


def fetch_signal_panel(cfg: Config) -> pd.DataFrame:
    """按候选字段顺序读取调仓截面数据；前一个字段不可用时自动尝试下一个。"""
    fetch_end = (
        pd.Timestamp(cfg.end_date) + pd.Timedelta(days=max(90, cfg.forward_days * 12))
    ).strftime("%Y-%m-%d")

    errors: List[str] = []
    chosen_source: Optional[FactorSource] = None
    df: Optional[pd.DataFrame] = None

    for src in cfg.factor_sources:
        try:
            tmp = _query_panel_with_source(cfg, src, fetch_end)
            if tmp.empty:
                errors.append(f"{src.table}.{src.field}: 查询成功但结果为空")
                continue
            chosen_source = src
            df = tmp
            break
        except Exception as e:
            short_err = str(e).split("\n")[-1]
            errors.append(f"{src.table}.{src.field}: {short_err}")
            continue

    if df is None or chosen_source is None:
        msg = [
            "qfa_roe 因子数据获取失败。已依次尝试以下 DAI 字段：",
            *errors,
            "",
            "建议先运行 test_factor_sources(CFG)，查看当前账号实际可用字段。",
            "注意：本代码不会自动回退到 roe_avg_lf 或 roe_avg_ttm，因为那会改变 qfa_roe 的单季度口径。",
        ]
        raise RuntimeError("\n".join(msg))

    print(f"qfa_roe 因子数据来源：{chosen_source.table}.{chosen_source.field}（{chosen_source.desc}）")

    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    df["instrument"] = df["instrument"].astype("category")
    for col in ["float_market_cap", "factor_raw", "ret_fwd", "excess_ret_fwd"]:
        df[col] = pd.to_numeric(df[col], errors="coerce", downcast="float")
    df = df.dropna(subset=["float_market_cap", "factor_raw", "ret_fwd", "excess_ret_fwd"])
    if df.empty:
        raise ValueError("清洗后数据为空，请检查日期区间、字段权限或股票池过滤条件。")
    return df.reset_index(drop=True)


def robust_zscore_np(x: np.ndarray) -> np.ndarray:
    """截面 MAD 去极值 + 标准化。按常见单因子测试写法使用 median ± 5 * MAD。"""
    x = x.astype(np.float64, copy=False)
    out = np.full(x.shape, np.nan, dtype=np.float64)
    valid = np.isfinite(x)
    if valid.sum() < 3:
        return out
    xv = x[valid]
    med = np.nanmedian(xv)
    mad = np.nanmedian(np.abs(xv - med))
    if np.isfinite(mad) and mad > 1e-12:
        lo, hi = med - 5.0 * mad, med + 5.0 * mad
    else:
        lo, hi = np.nanpercentile(xv, [1.0, 99.0])
    xv = np.clip(xv, lo, hi)
    std = xv.std(ddof=0)
    if np.isfinite(std) and std > 1e-12:
        out[valid] = (xv - xv.mean()) / std
    return out


def add_raw_factor_z(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    factor_z = np.full(len(df), np.nan, dtype=np.float32)
    raw = df["factor_raw"].to_numpy(dtype=np.float64, copy=False)
    for _, idx in df.groupby("date", sort=False, observed=True).indices.items():
        idx_arr = np.asarray(idx)
        factor_z[idx_arr] = robust_zscore_np(raw[idx_arr]).astype(np.float32)
    df["factor_z"] = factor_z
    return df.dropna(subset=["factor_z"]).reset_index(drop=True)


def corr_np(x: np.ndarray, y: np.ndarray) -> float:
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() < 3:
        return np.nan
    xv = x[valid].astype(np.float64, copy=False)
    yv = y[valid].astype(np.float64, copy=False)
    xv = xv - xv.mean()
    yv = yv - yv.mean()
    denom = np.sqrt(np.dot(xv, xv) * np.dot(yv, yv))
    if not np.isfinite(denom) or denom <= 1e-18:
        return np.nan
    return float(np.dot(xv, yv) / denom)


def rank_np(x: np.ndarray) -> np.ndarray:
    return pd.Series(x).rank(method="average").to_numpy(dtype=np.float64, copy=False)


def wls_factor_return(g: pd.DataFrame) -> Tuple[float, float]:
    y = g["excess_ret_fwd"].to_numpy(dtype=np.float64, copy=False)
    f = g["factor_z"].to_numpy(dtype=np.float64, copy=False)
    mv = g["float_market_cap"].to_numpy(dtype=np.float64, copy=False)
    w = np.sqrt(np.clip(mv, 1.0, None))
    X = np.column_stack([np.ones(len(g), dtype=np.float64), f])
    valid = np.isfinite(y) & np.isfinite(X).all(axis=1) & np.isfinite(w) & (w > 0)
    if valid.sum() < max(30, X.shape[1] + 5):
        return np.nan, np.nan
    Xv = X[valid]
    yv = y[valid]
    wv = w[valid]
    xtwx = Xv.T @ (wv[:, None] * Xv)
    xtwy = Xv.T @ (wv * yv)
    xtwx_inv = np.linalg.pinv(xtwx, rcond=1e-12)
    beta = xtwx_inv @ xtwy
    resid = yv - Xv @ beta
    rank = np.linalg.matrix_rank(xtwx)
    dof = max(len(yv) - rank, 1)
    sigma2 = float(np.sum(wv * resid * resid) / dof)
    se = np.sqrt(np.maximum(np.diag(sigma2 * xtwx_inv), 0.0))
    factor_ret = float(beta[1])
    t_value = float(beta[1] / se[1]) if se[1] > 1e-18 else np.nan
    return factor_ret, t_value


def calc_cross_section_metrics(g: pd.DataFrame, min_n: int) -> Optional[Dict[str, float]]:
    if len(g) < min_n:
        return None
    factor = g["factor_z"].to_numpy(dtype=np.float64, copy=False)
    ret = g["ret_fwd"].to_numpy(dtype=np.float64, copy=False)
    valid = np.isfinite(factor) & np.isfinite(ret)
    if valid.sum() < min_n:
        return None
    ic = corr_np(factor[valid], ret[valid])
    rank_ic = corr_np(rank_np(factor[valid]), rank_np(ret[valid]))
    factor_ret, t_value = wls_factor_return(g)
    return {
        "date": g["date"].iloc[0],
        "样本数": int(valid.sum()),
        "IC": ic,
        "RankIC": rank_ic,
        "因子收益率": factor_ret,
        "t值": t_value,
    }


def calc_all_metrics(data: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    rows: List[Dict[str, float]] = []
    for _, g in data.groupby("date", sort=True, observed=True):
        row = calc_cross_section_metrics(g, cfg.min_cross_section_size)
        if row is not None:
            rows.append(row)
    if not rows:
        raise ValueError("没有足够截面可计算指标，请检查区间、股票池或 min_cross_section_size。")
    return pd.DataFrame(rows).sort_values("date").reset_index(drop=True)


def safe_ir(s: pd.Series) -> float:
    s = pd.to_numeric(s, errors="coerce").dropna()
    std = s.std(ddof=1)
    if len(s) < 2 or not np.isfinite(std) or std <= 1e-18:
        return np.nan
    return float(s.mean() / std)


def make_summary(metrics: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    ic = pd.to_numeric(metrics["IC"], errors="coerce")
    rank_ic = pd.to_numeric(metrics["RankIC"], errors="coerce")
    factor_ret = pd.to_numeric(metrics["因子收益率"], errors="coerce")
    t_value = pd.to_numeric(metrics["t值"], errors="coerce")
    return pd.DataFrame([{
        "因子": cfg.factor_name,
        "起始日": metrics["date"].min().strftime("%Y-%m-%d"),
        "结束日": metrics["date"].max().strftime("%Y-%m-%d"),
        "截面数": int(metrics["date"].nunique()),
        "平均截面样本数": metrics["样本数"].mean(),
        "IC均值": ic.mean(),
        "IC标准差": ic.std(ddof=1),
        "ICIR": safe_ir(ic),
        "IC胜率": (ic > 0).mean(),
        "RankIC均值": rank_ic.mean(),
        "RankIC标准差": rank_ic.std(ddof=1),
        "RankICIR": safe_ir(rank_ic),
        "RankIC胜率": (rank_ic > 0).mean(),
        "因子收益率均值": factor_ret.mean(),
        "因子收益率标准差": factor_ret.std(ddof=1),
        "t值均值": t_value.mean(),
        "|t|均值": t_value.abs().mean(),
        "|t|>2占比": (t_value.abs() > 2).mean(),
        "t均值/t标准差": safe_ir(t_value),
    }])


def format_summary(summary: pd.DataFrame) -> pd.DataFrame:
    out = summary.copy()
    for col in ["截面数"]:
        out[col] = out[col].map(lambda x: "" if pd.isna(x) else f"{int(x)}")
    decimal_cols = [
        "平均截面样本数", "IC均值", "IC标准差", "ICIR", "RankIC均值", "RankIC标准差",
        "RankICIR", "因子收益率均值", "因子收益率标准差", "t值均值", "|t|均值", "t均值/t标准差",
    ]
    pct_cols = ["IC胜率", "RankIC胜率", "|t|>2占比"]
    for col in decimal_cols:
        out[col] = out[col].map(lambda x: "" if pd.isna(x) else f"{x:.6f}")
    for col in pct_cols:
        out[col] = out[col].map(lambda x: "" if pd.isna(x) else f"{x:.2%}")
    return out


def plot_ic_rankic(metrics: pd.DataFrame, cfg: Config) -> None:
    font_prop = set_chinese_font(cfg.chinese_font_path)
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(metrics["date"], metrics["IC"], label="IC", linewidth=1.6)
    ax.plot(metrics["date"], metrics["RankIC"], label="RankIC", linewidth=1.6)
    ax.axhline(0, linewidth=1.0, linestyle="--")
    title = f"{cfg.factor_name}：IC 与 RankIC 时序图"
    if font_prop is not None:
        ax.set_title(title, fontproperties=font_prop)
        ax.set_xlabel("日期", fontproperties=font_prop)
        ax.set_ylabel("相关系数", fontproperties=font_prop)
        ax.legend(prop=font_prop)
    else:
        ax.set_title(title)
        ax.set_xlabel("日期")
        ax.set_ylabel("相关系数")
        ax.legend()
    ax.grid(True, alpha=0.3)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()


def plot_cumsum_ic_rankic(metrics: pd.DataFrame, cfg: Config) -> None:
    font_prop = set_chinese_font(cfg.chinese_font_path)
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(metrics["date"], metrics["IC"].fillna(0).cumsum(), label="IC累计值", linewidth=1.6)
    ax.plot(metrics["date"], metrics["RankIC"].fillna(0).cumsum(), label="RankIC累计值", linewidth=1.6)
    ax.axhline(0, linewidth=1.0, linestyle="--")
    title = f"{cfg.factor_name}：IC 与 RankIC 累计曲线"
    if font_prop is not None:
        ax.set_title(title, fontproperties=font_prop)
        ax.set_xlabel("日期", fontproperties=font_prop)
        ax.set_ylabel("累计相关系数", fontproperties=font_prop)
        ax.legend(prop=font_prop)
    else:
        ax.set_title(title)
        ax.set_xlabel("日期")
        ax.set_ylabel("累计相关系数")
        ax.legend()
    ax.grid(True, alpha=0.3)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()


def main() -> Tuple[pd.DataFrame, pd.DataFrame]:
    data = fetch_signal_panel(CFG)
    data = add_raw_factor_z(data)
    metrics = calc_all_metrics(data, CFG)
    summary = make_summary(metrics, CFG)
    display(format_summary(summary))
    plot_ic_rankic(metrics, CFG)
    plot_cumsum_ic_rankic(metrics, CFG)
    return summary, metrics


summary, metrics = main()


## 市值行业中性化因子指标计算

In [ ]:
# -*- coding: utf-8 -*-
"""
BigQuant / DAI 复现华泰财务质量因子 qfa_roe：市值行业中性化因子指标测试版 v1。

口径说明：
1. 研报 qfa_roe = 单季度 ROE（平均），即单季度净利润 / 期初期末平均净资产。
2. BigQuant DAI 中 qfa_roe 的等价字段优先使用：roe_avg_mrq，字段含义为
   “净资产收益率(平均)(单季度)”。
3. 本代码只使用 dai.query，不调用旧版 D.features / D.financial_statements。
4. 因子处理流程：
   原始 qfa_roe -> 截面 MAD 去极值 + 标准化 -> 对 log(流通市值) 与行业哑变量做截面 OLS 回归
   -> 取残差 -> 再次截面标准化，得到 qfa_roe_neutral。
5. 行业默认使用 cn_stock_factors_industry.sw2021_level1，即申万 2021 一级行业；如需更细，
   可把 Config.industry_sources 中 sw2021_level2 放到第一位。
6. 每隔 rebalance_freq 个交易日取一个截面，计算未来 forward_days 个交易日收益的
   IC、RankIC、WLS 因子收益率和 t 值。
7. 股票池剔除风险警示/ST、当前停牌、下一交易日停牌、北交所；收益使用日收盘价。
8. 回归法：未来 forward_days 日相对沪深300超额收益 ~ const + qfa_roe_neutral，
   WLS 权重为 sqrt(流通市值)。注意这里的权重只用于估计稳定性，中性化已在因子端完成。
9. 若报字段不存在，请先运行：
       test_data_sources(CFG)
   检查当前 BigQuant / DAI 环境中 qfa_roe 字段与行业字段是否可读。

输出说明：
- summary：整体指标汇总。
- metrics：逐截面 IC、RankIC、因子收益率、t 值。
- diagnostics：逐截面中性化诊断，用于观察中性化后因子与 log(流通市值) 的相关性。
"""

import os
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import display

try:
    import dai
except ImportError:
    from bigquant import dai  # type: ignore

warnings.filterwarnings("ignore")


@dataclass(frozen=True)
class FactorSource:
    table: str
    field: str
    desc: str


@dataclass(frozen=True)
class IndustrySource:
    table: str
    field: str
    desc: str


@dataclass(frozen=True)
class Config:
    start_date: str = "2020-01-01"
    end_date: str = "2026-06-30"

    forward_days: int = 30
    rebalance_freq: int = 30
    min_cross_section_size: int = 30
    min_neutralize_size: int = 30
    min_industry_count: int = 10

    factor_name: str = "qfa_roe_neutral"
    chinese_font_path: str = ""

    # qfa_roe 的 BigQuant 等价字段：roe_avg_mrq = 净资产收益率(平均)(单季度)
    # 不回退到 roe_avg_lf / roe_avg_ttm，避免改变因子口径。
    factor_sources: Tuple[FactorSource, ...] = field(default_factory=lambda: (
        FactorSource("cn_stock_prefactors", "roe_avg_mrq", "净资产收益率(平均)(单季度)，对应研报 qfa_roe"),
        FactorSource("cn_stock_factors_financial_indicators", "roe_avg_mrq", "净资产收益率(平均)(单季度)，对应研报 qfa_roe"),
    ))

    # 默认申万2021一级行业；若希望行业约束更细，可把 sw2021_level2 放到第一位。
    industry_sources: Tuple[IndustrySource, ...] = field(default_factory=lambda: (
        IndustrySource("cn_stock_factors_industry", "sw2021_level1", "申万2021一级行业代码"),
        IndustrySource("cn_stock_factors_industry", "sw2021_level2", "申万2021二级行业代码"),
        IndustrySource("cn_stock_factors_industry", "cs_level1", "中信一级行业代码"),
    ))


CFG = Config()


def test_data_sources(cfg: Config = CFG) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """测试 qfa_roe 字段和行业字段是否可在当前 BigQuant / DAI 环境中读取。"""
    factor_rows: List[Dict[str, object]] = []
    for src in cfg.factor_sources:
        sql = f"""
        SELECT date, instrument, {src.field} AS factor_raw
        FROM {src.table}
        WHERE date >= DATE '{cfg.start_date}'
          AND date <= DATE '{cfg.end_date}'
          AND {src.field} IS NOT NULL
        LIMIT 5
        """
        try:
            tmp = dai.query(sql, filters={"date": [cfg.start_date, cfg.end_date]}).df()
            ok = not tmp.empty
            factor_rows.append({
                "table": src.table,
                "field": src.field,
                "desc": src.desc,
                "available": ok,
                "rows": len(tmp),
                "error": "" if ok else "查询成功但无非空样本",
            })
        except Exception as e:
            factor_rows.append({
                "table": src.table,
                "field": src.field,
                "desc": src.desc,
                "available": False,
                "rows": 0,
                "error": str(e).split("\n")[-1][:240],
            })

    industry_rows: List[Dict[str, object]] = []
    for src in cfg.industry_sources:
        sql = f"""
        SELECT date, instrument, {src.field} AS industry_code
        FROM {src.table}
        WHERE date >= DATE '{cfg.start_date}'
          AND date <= DATE '{cfg.end_date}'
          AND {src.field} IS NOT NULL
        LIMIT 5
        """
        try:
            tmp = dai.query(sql, filters={"date": [cfg.start_date, cfg.end_date]}).df()
            ok = not tmp.empty
            industry_rows.append({
                "table": src.table,
                "field": src.field,
                "desc": src.desc,
                "available": ok,
                "rows": len(tmp),
                "error": "" if ok else "查询成功但无非空样本",
            })
        except Exception as e:
            industry_rows.append({
                "table": src.table,
                "field": src.field,
                "desc": src.desc,
                "available": False,
                "rows": 0,
                "error": str(e).split("\n")[-1][:240],
            })

    factor_out = pd.DataFrame(factor_rows)
    industry_out = pd.DataFrame(industry_rows)
    print("qfa_roe 字段可用性：")
    display(factor_out)
    print("行业字段可用性：")
    display(industry_out)
    return factor_out, industry_out


# 为了兼容上一版代码里的函数名，保留这个别名。
def test_factor_sources(cfg: Config = CFG) -> Tuple[pd.DataFrame, pd.DataFrame]:
    return test_data_sources(cfg)


def _font_has_chinese(font_path: str) -> bool:
    try:
        ft = font_manager.get_font(font_path)
        cmap = ft.get_charmap()
        return all(ord(ch) in cmap for ch in "因子日期相关系数")
    except Exception:
        return False


def _candidate_font_paths(user_font_path: str = "") -> List[str]:
    candidates: List[str] = []
    if user_font_path:
        candidates.append(user_font_path)
    env_font = os.environ.get("CHINESE_FONT_PATH", "")
    if env_font:
        candidates.append(env_font)
    candidates.extend([
        "./SimHei.ttf", "./simhei.ttf", "./msyh.ttc", "./Microsoft YaHei.ttf",
        "./NotoSansCJK-Regular.ttc", "/home/jovyan/work/SimHei.ttf",
        "/home/jovyan/work/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/truetype/wqy/wqy-microhei.ttc",
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/opentype/noto/NotoSansCJKsc-Regular.otf",
        "/System/Library/Fonts/PingFang.ttc",
        "C:/Windows/Fonts/msyh.ttc", "C:/Windows/Fonts/simhei.ttf",
    ])

    search_roots = [Path.cwd(), Path.home(), Path("/usr/share/fonts"), Path("/usr/local/share/fonts")]
    name_keywords = (
        "NotoSansCJK", "NotoSansSC", "SourceHanSans", "WenQuanYi", "wqy",
        "SimHei", "simhei", "msyh", "PingFang", "Arial Unicode",
    )
    suffixes = {".ttf", ".ttc", ".otf"}
    for root in search_roots:
        if not root.exists():
            continue
        try:
            for p in root.rglob("*"):
                if p.suffix.lower() in suffixes and any(k.lower() in p.name.lower() for k in name_keywords):
                    candidates.append(str(p))
        except Exception:
            continue

    seen = set()
    unique = []
    for p in candidates:
        pp = str(Path(p).expanduser())
        if pp not in seen:
            seen.add(pp)
            unique.append(pp)
    return unique


def set_chinese_font(font_path: str = "") -> Optional[font_manager.FontProperties]:
    preferred_names = [
        "Microsoft YaHei", "SimHei", "Noto Sans CJK SC", "Noto Sans SC",
        "Source Han Sans SC", "WenQuanYi Micro Hei", "PingFang SC", "Arial Unicode MS",
    ]
    plt.rcParams["axes.unicode_minus"] = False
    for path in _candidate_font_paths(font_path):
        if not Path(path).exists() or not _font_has_chinese(path):
            continue
        try:
            font_manager.fontManager.addfont(path)
            prop = font_manager.FontProperties(fname=path)
            font_name = prop.get_name()
            plt.rcParams["font.family"] = "sans-serif"
            plt.rcParams["font.sans-serif"] = [font_name] + preferred_names + ["DejaVu Sans"]
            return prop
        except Exception:
            continue

    available = {f.name for f in font_manager.fontManager.ttflist}
    for name in preferred_names:
        if name in available:
            plt.rcParams["font.family"] = "sans-serif"
            plt.rcParams["font.sans-serif"] = [name] + ["DejaVu Sans"]
            return font_manager.FontProperties(family=name)

    plt.rcParams["font.family"] = "sans-serif"
    plt.rcParams["font.sans-serif"] = preferred_names + ["DejaVu Sans"]
    return None


def _query_panel_with_sources(
    cfg: Config,
    factor_src: FactorSource,
    industry_src: IndustrySource,
    fetch_end: str,
) -> pd.DataFrame:
    """使用指定因子字段与行业字段读取完整测试面板。"""
    if factor_src.table == "cn_stock_prefactors":
        factor_join = ""
        factor_select = "b.factor_from_base AS factor_raw"
        factor_filter = "b.factor_from_base IS NOT NULL"
        base_factor_expr = factor_src.field
    else:
        factor_join = f"""
        JOIN {factor_src.table} AS f
          ON b.date = f.date AND b.instrument = f.instrument
        """
        factor_select = f"f.{factor_src.field} AS factor_raw"
        factor_filter = f"f.{factor_src.field} IS NOT NULL"
        base_factor_expr = "CAST(NULL AS DOUBLE)"

    sql = f"""
    PRAGMA enable_pushdown_window;

    WITH trading_dates AS (
        SELECT
            date,
            ROW_NUMBER() OVER (ORDER BY date) AS rn
        FROM (
            SELECT DISTINCT date
            FROM cn_stock_prefactors
            WHERE date >= DATE '{cfg.start_date}'
              AND date <= DATE '{cfg.end_date}'
        )
    ),

    signal_dates AS (
        SELECT date
        FROM trading_dates
        WHERE MOD(rn - 1, {cfg.rebalance_freq}) = 0
    ),

    base AS (
        SELECT
            date,
            instrument,
            close,
            float_market_cap,
            is_risk_warning,
            suspended,
            list_sector,
            {base_factor_expr} AS factor_from_base,
            LEAD(close, {cfg.forward_days}) OVER (
                PARTITION BY instrument ORDER BY date
            ) AS close_fwd,
            LEAD(suspended, 1) OVER (
                PARTITION BY instrument ORDER BY date
            ) AS next_suspended
        FROM cn_stock_prefactors
        WHERE date >= DATE '{cfg.start_date}'
          AND date <= DATE '{fetch_end}'
          AND COALESCE(list_sector, 0) != 4
    ),

    bench_raw AS (
        SELECT
            date,
            MAX(close_000300SH) AS hs300_close
        FROM cn_stock_prefactors
        WHERE date >= DATE '{cfg.start_date}'
          AND date <= DATE '{fetch_end}'
        GROUP BY date
    ),

    bench AS (
        SELECT
            date,
            hs300_close,
            LEAD(hs300_close, {cfg.forward_days}) OVER (ORDER BY date) AS hs300_close_fwd
        FROM bench_raw
    )

    SELECT
        b.date,
        b.instrument,
        b.float_market_cap,
        ind.{industry_src.field} AS industry_code,
        {factor_select},
        b.close_fwd / b.close - 1.0 AS ret_fwd,
        b.close_fwd / b.close - 1.0
            - (be.hs300_close_fwd / be.hs300_close - 1.0) AS excess_ret_fwd
    FROM base AS b
    JOIN signal_dates AS sd
      ON b.date = sd.date
    {factor_join}
    JOIN {industry_src.table} AS ind
      ON b.date = ind.date AND b.instrument = ind.instrument
    JOIN bench AS be
      ON b.date = be.date
    WHERE b.date <= DATE '{cfg.end_date}'
      AND b.is_risk_warning = 0
      AND b.suspended = 0
      AND COALESCE(b.next_suspended, 1) = 0
      AND b.close > 0
      AND b.close_fwd > 0
      AND be.hs300_close > 0
      AND be.hs300_close_fwd > 0
      AND b.float_market_cap > 0
      AND {factor_filter}
      AND ind.{industry_src.field} IS NOT NULL
    ORDER BY b.date, b.instrument
    """
    return dai.query(sql, filters={"date": [cfg.start_date, fetch_end]}).df()


def fetch_signal_panel(cfg: Config) -> pd.DataFrame:
    """按候选字段顺序读取调仓截面数据；字段不可用时自动尝试下一个。"""
    fetch_end = (
        pd.Timestamp(cfg.end_date) + pd.Timedelta(days=max(90, cfg.forward_days * 12))
    ).strftime("%Y-%m-%d")

    errors: List[str] = []
    chosen_factor: Optional[FactorSource] = None
    chosen_industry: Optional[IndustrySource] = None
    df: Optional[pd.DataFrame] = None

    for factor_src in cfg.factor_sources:
        for industry_src in cfg.industry_sources:
            try:
                tmp = _query_panel_with_sources(cfg, factor_src, industry_src, fetch_end)
                if tmp.empty:
                    errors.append(f"{factor_src.table}.{factor_src.field} + {industry_src.table}.{industry_src.field}: 查询成功但结果为空")
                    continue
                chosen_factor = factor_src
                chosen_industry = industry_src
                df = tmp
                break
            except Exception as e:
                short_err = str(e).split("\n")[-1]
                errors.append(f"{factor_src.table}.{factor_src.field} + {industry_src.table}.{industry_src.field}: {short_err}")
                continue
        if df is not None:
            break

    if df is None or chosen_factor is None or chosen_industry is None:
        msg = [
            "qfa_roe 市值行业中性化数据获取失败。已依次尝试以下组合：",
            *errors,
            "",
            "建议先运行 test_data_sources(CFG)，查看当前账号实际可用字段。",
            "注意：本代码不会自动回退到 roe_avg_lf 或 roe_avg_ttm，因为那会改变 qfa_roe 的单季度口径。",
        ]
        raise RuntimeError("\n".join(msg))

    print(f"qfa_roe 因子数据来源：{chosen_factor.table}.{chosen_factor.field}（{chosen_factor.desc}）")
    print(f"行业中性化字段：{chosen_industry.table}.{chosen_industry.field}（{chosen_industry.desc}）")

    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    df["instrument"] = df["instrument"].astype("category")
    df["industry_code"] = df["industry_code"].astype(str).replace({"": np.nan, "None": np.nan, "nan": np.nan})
    for col in ["float_market_cap", "factor_raw", "ret_fwd", "excess_ret_fwd"]:
        df[col] = pd.to_numeric(df[col], errors="coerce", downcast="float")
    df = df.dropna(subset=["float_market_cap", "industry_code", "factor_raw", "ret_fwd", "excess_ret_fwd"])
    if df.empty:
        raise ValueError("清洗后数据为空，请检查日期区间、字段权限、行业字段或股票池过滤条件。")
    return df.reset_index(drop=True)


def robust_zscore_np(x: np.ndarray) -> np.ndarray:
    """截面 MAD 去极值 + 标准化。按常见单因子测试写法使用 median ± 5 * MAD。"""
    x = x.astype(np.float64, copy=False)
    out = np.full(x.shape, np.nan, dtype=np.float64)
    valid = np.isfinite(x)
    if valid.sum() < 3:
        return out
    xv = x[valid]
    med = np.nanmedian(xv)
    mad = np.nanmedian(np.abs(xv - med))
    if np.isfinite(mad) and mad > 1e-12:
        lo, hi = med - 5.0 * mad, med + 5.0 * mad
    else:
        lo, hi = np.nanpercentile(xv, [1.0, 99.0])
    xv = np.clip(xv, lo, hi)
    std = xv.std(ddof=0)
    if np.isfinite(std) and std > 1e-12:
        out[valid] = (xv - xv.mean()) / std
    return out


def _prepare_industry_labels(industry: pd.Series, min_count: int) -> pd.Series:
    """把过小行业合并为 OTHER，避免哑变量矩阵过于稀疏。"""
    labels = industry.astype(str).replace({"": "UNKNOWN", "None": "UNKNOWN", "nan": "UNKNOWN"})
    counts = labels.value_counts(dropna=False)
    small = counts[counts < min_count].index
    return labels.where(~labels.isin(small), "OTHER")


def neutralize_one_cross_section(g: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    """单个截面内对 qfa_roe 做市值和行业中性化。"""
    out = g.copy()
    out["factor_raw_z"] = np.nan
    out["factor_z"] = np.nan
    out["neutral_mv_corr"] = np.nan

    raw = out["factor_raw"].to_numpy(dtype=np.float64, copy=False)
    raw_z = robust_zscore_np(raw)
    out["factor_raw_z"] = raw_z.astype(np.float32)

    mv = out["float_market_cap"].to_numpy(dtype=np.float64, copy=False)
    log_mv = np.log(np.clip(mv, 1.0, None))
    industry = _prepare_industry_labels(out["industry_code"], cfg.min_industry_count)

    valid = np.isfinite(raw_z) & np.isfinite(log_mv) & industry.notna().to_numpy()
    if valid.sum() < cfg.min_neutralize_size:
        return out

    y = raw_z[valid]
    mv_x = log_mv[valid]
    mv_x = mv_x - np.nanmean(mv_x)

    ind_valid = industry.iloc[np.where(valid)[0]]
    dummies = pd.get_dummies(ind_valid, prefix="ind", drop_first=True, dtype=float)
    if dummies.shape[1] == 0:
        X = np.column_stack([np.ones(len(y), dtype=np.float64), mv_x])
    else:
        X = np.column_stack([np.ones(len(y), dtype=np.float64), mv_x, dummies.to_numpy(dtype=np.float64)])

    # 若行业哑变量过多且有效样本不足，回退为仅对市值中性化，避免过拟合或奇异矩阵。
    if len(y) <= X.shape[1] + 5:
        X = np.column_stack([np.ones(len(y), dtype=np.float64), mv_x])

    beta = np.linalg.pinv(X, rcond=1e-12) @ y
    resid = y - X @ beta

    resid_z = robust_zscore_np(resid)
    valid_indices = np.where(valid)[0]
    out.iloc[valid_indices, out.columns.get_loc("factor_z")] = resid_z.astype(np.float32)

    mv_corr = corr_np(resid_z, mv_x)
    out["neutral_mv_corr"] = mv_corr
    return out


def add_neutral_factor_z(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    """逐截面做 MAD、标准化、市值行业中性化，并返回中性化后的 factor_z。"""
    parts: List[pd.DataFrame] = []
    for _, g in df.groupby("date", sort=False, observed=True):
        parts.append(neutralize_one_cross_section(g, cfg))
    out = pd.concat(parts, axis=0, ignore_index=True)
    out = out.dropna(subset=["factor_z"]).reset_index(drop=True)
    if out.empty:
        raise ValueError("市值行业中性化后数据为空，请检查行业字段、样本数或 min_neutralize_size。")
    return out


# 为了兼容裸因子版 main 中的函数名，保留这个别名；这里实际执行的是中性化。
def add_raw_factor_z(df: pd.DataFrame) -> pd.DataFrame:
    return add_neutral_factor_z(df, CFG)


def corr_np(x: np.ndarray, y: np.ndarray) -> float:
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() < 3:
        return np.nan
    xv = x[valid].astype(np.float64, copy=False)
    yv = y[valid].astype(np.float64, copy=False)
    xv = xv - xv.mean()
    yv = yv - yv.mean()
    denom = np.sqrt(np.dot(xv, xv) * np.dot(yv, yv))
    if not np.isfinite(denom) or denom <= 1e-18:
        return np.nan
    return float(np.dot(xv, yv) / denom)


def rank_np(x: np.ndarray) -> np.ndarray:
    return pd.Series(x).rank(method="average").to_numpy(dtype=np.float64, copy=False)


def wls_factor_return(g: pd.DataFrame) -> Tuple[float, float]:
    y = g["excess_ret_fwd"].to_numpy(dtype=np.float64, copy=False)
    f = g["factor_z"].to_numpy(dtype=np.float64, copy=False)
    mv = g["float_market_cap"].to_numpy(dtype=np.float64, copy=False)
    w = np.sqrt(np.clip(mv, 1.0, None))
    X = np.column_stack([np.ones(len(g), dtype=np.float64), f])
    valid = np.isfinite(y) & np.isfinite(X).all(axis=1) & np.isfinite(w) & (w > 0)
    if valid.sum() < max(30, X.shape[1] + 5):
        return np.nan, np.nan
    Xv = X[valid]
    yv = y[valid]
    wv = w[valid]
    xtwx = Xv.T @ (wv[:, None] * Xv)
    xtwy = Xv.T @ (wv * yv)
    xtwx_inv = np.linalg.pinv(xtwx, rcond=1e-12)
    beta = xtwx_inv @ xtwy
    resid = yv - Xv @ beta
    rank = np.linalg.matrix_rank(xtwx)
    dof = max(len(yv) - rank, 1)
    sigma2 = float(np.sum(wv * resid * resid) / dof)
    se = np.sqrt(np.maximum(np.diag(sigma2 * xtwx_inv), 0.0))
    factor_ret = float(beta[1])
    t_value = float(beta[1] / se[1]) if se[1] > 1e-18 else np.nan
    return factor_ret, t_value


def calc_cross_section_metrics(g: pd.DataFrame, min_n: int) -> Optional[Dict[str, float]]:
    if len(g) < min_n:
        return None
    factor = g["factor_z"].to_numpy(dtype=np.float64, copy=False)
    ret = g["ret_fwd"].to_numpy(dtype=np.float64, copy=False)
    valid = np.isfinite(factor) & np.isfinite(ret)
    if valid.sum() < min_n:
        return None
    ic = corr_np(factor[valid], ret[valid])
    rank_ic = corr_np(rank_np(factor[valid]), rank_np(ret[valid]))
    factor_ret, t_value = wls_factor_return(g)
    return {
        "date": g["date"].iloc[0],
        "样本数": int(valid.sum()),
        "IC": ic,
        "RankIC": rank_ic,
        "因子收益率": factor_ret,
        "t值": t_value,
    }


def calc_all_metrics(data: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    rows: List[Dict[str, float]] = []
    for _, g in data.groupby("date", sort=True, observed=True):
        row = calc_cross_section_metrics(g, cfg.min_cross_section_size)
        if row is not None:
            rows.append(row)
    if not rows:
        raise ValueError("没有足够截面可计算指标，请检查区间、股票池或 min_cross_section_size。")
    return pd.DataFrame(rows).sort_values("date").reset_index(drop=True)


def calc_neutralization_diagnostics(data: pd.DataFrame) -> pd.DataFrame:
    """检查中性化后因子和 log(流通市值) 的相关性。"""
    rows: List[Dict[str, float]] = []
    for dt, g in data.groupby("date", sort=True, observed=True):
        f = g["factor_z"].to_numpy(dtype=np.float64, copy=False)
        raw = g["factor_raw_z"].to_numpy(dtype=np.float64, copy=False)
        log_mv = np.log(np.clip(g["float_market_cap"].to_numpy(dtype=np.float64, copy=False), 1.0, None))
        rows.append({
            "date": dt,
            "样本数": len(g),
            "中性化前因子与log市值相关": corr_np(raw, log_mv),
            "中性化后因子与log市值相关": corr_np(f, log_mv),
            "行业数量": int(g["industry_code"].nunique()),
        })
    return pd.DataFrame(rows)


def safe_ir(s: pd.Series) -> float:
    s = pd.to_numeric(s, errors="coerce").dropna()
    std = s.std(ddof=1)
    if len(s) < 2 or not np.isfinite(std) or std <= 1e-18:
        return np.nan
    return float(s.mean() / std)


def make_summary(metrics: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    ic = pd.to_numeric(metrics["IC"], errors="coerce")
    rank_ic = pd.to_numeric(metrics["RankIC"], errors="coerce")
    factor_ret = pd.to_numeric(metrics["因子收益率"], errors="coerce")
    t_value = pd.to_numeric(metrics["t值"], errors="coerce")
    return pd.DataFrame([{
        "因子": cfg.factor_name,
        "起始日": metrics["date"].min().strftime("%Y-%m-%d"),
        "结束日": metrics["date"].max().strftime("%Y-%m-%d"),
        "截面数": int(metrics["date"].nunique()),
        "平均截面样本数": metrics["样本数"].mean(),
        "IC均值": ic.mean(),
        "IC标准差": ic.std(ddof=1),
        "ICIR": safe_ir(ic),
        "IC胜率": (ic > 0).mean(),
        "RankIC均值": rank_ic.mean(),
        "RankIC标准差": rank_ic.std(ddof=1),
        "RankICIR": safe_ir(rank_ic),
        "RankIC胜率": (rank_ic > 0).mean(),
        "因子收益率均值": factor_ret.mean(),
        "因子收益率标准差": factor_ret.std(ddof=1),
        "t值均值": t_value.mean(),
        "|t|均值": t_value.abs().mean(),
        "|t|>2占比": (t_value.abs() > 2).mean(),
        "t均值/t标准差": safe_ir(t_value),
    }])


def make_neutralization_summary(diagnostics: pd.DataFrame) -> pd.DataFrame:
    before = pd.to_numeric(diagnostics["中性化前因子与log市值相关"], errors="coerce")
    after = pd.to_numeric(diagnostics["中性化后因子与log市值相关"], errors="coerce")
    return pd.DataFrame([{
        "截面数": diagnostics["date"].nunique(),
        "平均行业数量": diagnostics["行业数量"].mean(),
        "中性化前市值相关均值": before.mean(),
        "中性化前市值相关绝对值均值": before.abs().mean(),
        "中性化后市值相关均值": after.mean(),
        "中性化后市值相关绝对值均值": after.abs().mean(),
    }])


def format_summary(summary: pd.DataFrame) -> pd.DataFrame:
    out = summary.copy()
    for col in ["截面数"]:
        out[col] = out[col].map(lambda x: "" if pd.isna(x) else f"{int(x)}")
    decimal_cols = [
        "平均截面样本数", "IC均值", "IC标准差", "ICIR", "RankIC均值", "RankIC标准差",
        "RankICIR", "因子收益率均值", "因子收益率标准差", "t值均值", "|t|均值", "t均值/t标准差",
        "平均行业数量", "中性化前市值相关均值", "中性化前市值相关绝对值均值",
        "中性化后市值相关均值", "中性化后市值相关绝对值均值",
    ]
    pct_cols = ["IC胜率", "RankIC胜率", "|t|>2占比"]
    for col in decimal_cols:
        if col in out.columns:
            out[col] = out[col].map(lambda x: "" if pd.isna(x) else f"{x:.6f}")
    for col in pct_cols:
        if col in out.columns:
            out[col] = out[col].map(lambda x: "" if pd.isna(x) else f"{x:.2%}")
    return out


def plot_ic_rankic(metrics: pd.DataFrame, cfg: Config) -> None:
    font_prop = set_chinese_font(cfg.chinese_font_path)
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(metrics["date"], metrics["IC"], label="IC", linewidth=1.6)
    ax.plot(metrics["date"], metrics["RankIC"], label="RankIC", linewidth=1.6)
    ax.axhline(0, linewidth=1.0, linestyle="--")
    title = f"{cfg.factor_name}：IC 与 RankIC 时序图"
    if font_prop is not None:
        ax.set_title(title, fontproperties=font_prop)
        ax.set_xlabel("日期", fontproperties=font_prop)
        ax.set_ylabel("相关系数", fontproperties=font_prop)
        ax.legend(prop=font_prop)
    else:
        ax.set_title(title)
        ax.set_xlabel("日期")
        ax.set_ylabel("相关系数")
        ax.legend()
    ax.grid(True, alpha=0.3)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()


def plot_cumsum_ic_rankic(metrics: pd.DataFrame, cfg: Config) -> None:
    font_prop = set_chinese_font(cfg.chinese_font_path)
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(metrics["date"], metrics["IC"].fillna(0).cumsum(), label="IC累计值", linewidth=1.6)
    ax.plot(metrics["date"], metrics["RankIC"].fillna(0).cumsum(), label="RankIC累计值", linewidth=1.6)
    ax.axhline(0, linewidth=1.0, linestyle="--")
    title = f"{cfg.factor_name}：IC 与 RankIC 累计曲线"
    if font_prop is not None:
        ax.set_title(title, fontproperties=font_prop)
        ax.set_xlabel("日期", fontproperties=font_prop)
        ax.set_ylabel("累计相关系数", fontproperties=font_prop)
        ax.legend(prop=font_prop)
    else:
        ax.set_title(title)
        ax.set_xlabel("日期")
        ax.set_ylabel("累计相关系数")
        ax.legend()
    ax.grid(True, alpha=0.3)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()


def main() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    data = fetch_signal_panel(CFG)
    data = add_neutral_factor_z(data, CFG)
    metrics = calc_all_metrics(data, CFG)
    summary = make_summary(metrics, CFG)
    diagnostics = calc_neutralization_diagnostics(data)
    neutral_summary = make_neutralization_summary(diagnostics)

    print("市值行业中性化后因子指标：")
    display(format_summary(summary))
    print("中性化诊断：")
    display(format_summary(neutral_summary))

    plot_ic_rankic(metrics, CFG)
    plot_cumsum_ic_rankic(metrics, CFG)
    return summary, metrics, diagnostics


summary, metrics, diagnostics = main()


## 市值行业中性化因子市值分组回测

In [ ]:
# -*- coding: utf-8 -*-
"""
BigQuant 策略回测：qfa_roe_neutral 市值分层选股策略

因子口径：
1. 研报 qfa_roe = 单季度 ROE（平均），BigQuant 中优先使用 cn_stock_prefactors.roe_avg_mrq。
2. qfa_roe 为正向因子，原始因子直接使用 roe_avg_mrq，不取相反数。
3. 每个信号截面：原始因子先做 MAD 去极值 + 标准化，再对 log(流通市值) 与行业做截面中性化，
   最后对中性化残差再次做 MAD 去极值 + 标准化，得到 qfa_roe_neutral。

策略逻辑：
1. 每 REBALANCE_DAYS 个交易日生成一次信号，信号使用 signal_date 当日已经可得的数据。
2. 全市场按流通市值从小到大划分 15 组，1=最小市值组，15=最大市值组。
3. 通过 SIZE_GROUPS_TO_TRADE 指定参与交易的市值组，例如 [1, 2, 3]。
4. 在每个指定市值组内，选取 qfa_roe_neutral 最高的前 TOP_PCT 股票。
5. 所有入选股票等权配置。
6. 信号日后一交易日执行调仓；不在选股阶段读取执行日行情，避免未来函数。
7. 执行日使用当日开盘涨跌停状态做交易约束：开盘涨停不买入，开盘跌停不卖出；停牌不交易。
8. 股票池剔除 ST、*ST、停牌、北交所；考虑交易成本；使用 BigTrader 原生回测引擎。

注意：
- 执行日的开盘涨跌停限制只在 handle_data 当日下单时使用，不参与因子计算与选股。
- 如果某只股票因涨停无法买入或因跌停无法卖出，不做强制再平衡，避免使用执行日信息重新分配权重。
"""

import gc
import time
import warnings
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

try:
    import dai
except Exception as e:
    raise ImportError("请在 BigQuant Notebook 环境中运行，本代码依赖 dai。") from e

try:
    from bigquant import bigtrader
except Exception:
    try:
        import bigtrader
    except Exception as e:
        raise ImportError("请在 BigQuant Notebook 环境中运行，本代码依赖 BigTrader。") from e

warnings.filterwarnings("ignore")


# =========================
# 1. 参数设置
# =========================

START_DATE = "2020-01-01"
END_DATE = "2026-06-30"

FACTOR_TABLE = "cn_stock_prefactors"
FACTOR_SOURCE_COL = "roe_avg_mrq"
RAW_FACTOR_NAME = "qfa_roe_raw"
NEUTRAL_FACTOR_NAME = "qfa_roe_neutral"

# 1=最小市值组，15=最大市值组
SIZE_GROUPS_TO_TRADE = [15]
N_SIZE_GROUPS = 15
TOP_PCT = 0.05
REBALANCE_DAYS = 30
MIN_STOCKS_PER_SIZE_GROUP = 20
SELECT_COUNT_METHOD = "floor"  # floor 或 ceil；至少选 1 只

# 中性化与标准化
WINSOR_MAD_N = 3.0
MIN_CROSS_SECTION_SIZE = 100
INDUSTRY_COL = "sw2021_level1"  # 可改 sw2014_level1 / cs_level1

# 回测参数
CAPITAL_BASE = 1_000_000
BENCHMARK = "000300.SH"  # 如账号环境使用 CSI 代码，可改为 000300.SH / 000300.SH / 932000.CSI 等
BUY_COST = 0.0003
SELL_COST = 0.0013
MIN_COMMISSION = 5

# 因子字段说明：qfa_roe 对应 BigQuant roe_avg_mrq，即净资产收益率(平均)(单季度)。
# 如当前账号字段权限异常，可先在 BigQuant 数据平台确认字段，或将 FACTOR_TABLE / FACTOR_SOURCE_COL 改为可用表字段。

# 执行日涨跌停判断容忍误差
LIMIT_EPS = 1e-4

START_DATE = pd.to_datetime(START_DATE).strftime("%Y-%m-%d")
END_DATE = pd.to_datetime(END_DATE).strftime("%Y-%m-%d")

if not SIZE_GROUPS_TO_TRADE:
    raise ValueError("SIZE_GROUPS_TO_TRADE 不能为空。")
SIZE_GROUPS_TO_TRADE = sorted(set(int(x) for x in SIZE_GROUPS_TO_TRADE))
bad_groups = [g for g in SIZE_GROUPS_TO_TRADE if g < 1 or g > N_SIZE_GROUPS]
if bad_groups:
    raise ValueError(f"SIZE_GROUPS_TO_TRADE 中存在非法市值组：{bad_groups}，有效范围为 1~{N_SIZE_GROUPS}。")
if SELECT_COUNT_METHOD not in {"floor", "ceil"}:
    raise ValueError("SELECT_COUNT_METHOD 只能是 'floor' 或 'ceil'。")


# =========================
# 2. 通用工具函数
# =========================

_T0 = time.time()


def _elapsed() -> str:
    sec = int(time.time() - _T0)
    return f"{sec // 60:02d}:{sec % 60:02d}"


def progress(msg: str) -> None:
    print(f"[{_elapsed()}] {msg}", flush=True)


def query_df(sql: str, filters: Optional[dict] = None) -> pd.DataFrame:
    if filters is None:
        return dai.query(sql).df()
    return dai.query(sql, filters=filters).df()




def test_data_sources() -> pd.DataFrame:
    """检查当前账号是否能读取本策略需要的核心字段。"""
    tests = [
        ("cn_stock_prefactors", "roe_avg_mrq", "qfa_roe 对应字段：净资产收益率(平均)(单季度)"),
        ("cn_stock_factors_base", "float_market_cap", "流通市值"),
        ("cn_stock_factors_base", INDUSTRY_COL, "行业字段"),
        ("cn_stock_factors_base", "st_status", "ST 状态"),
        ("cn_stock_factors_base", "suspended", "停牌状态"),
        ("cn_stock_factors_base", "open", "执行日开盘价"),
        ("cn_stock_factors_base", "upper_limit", "执行日涨停价"),
        ("cn_stock_factors_base", "lower_limit", "执行日跌停价"),
    ]
    rows = []
    for table, field, desc in tests:
        sql = f"""
        SELECT date, instrument, {field} AS value
        FROM {table}
        WHERE date >= DATE '{START_DATE}'
          AND date <= DATE '{END_DATE}'
          AND {field} IS NOT NULL
        LIMIT 5
        """
        try:
            tmp = query_df(sql, filters={"date": [START_DATE, END_DATE]})
            rows.append({
                "table": table,
                "field": field,
                "desc": desc,
                "available": not tmp.empty,
                "rows": len(tmp),
                "error": "" if not tmp.empty else "查询成功但无非空样本",
            })
        except Exception as e:
            rows.append({
                "table": table,
                "field": field,
                "desc": desc,
                "available": False,
                "rows": 0,
                "error": str(e).split("\n")[-1][:240],
            })
    out = pd.DataFrame(rows)
    display(out)
    return out

def date_in_sql(dates) -> str:
    return ", ".join(f"'{pd.to_datetime(x).strftime('%Y-%m-%d')}'" for x in dates)


def instrument_in_sql(instruments: List[str]) -> str:
    return ", ".join(f"'{str(x)}'" for x in instruments)


def downcast_float(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce", downcast="float")
    return df


def robust_zscore_np(x: np.ndarray, mad_n: float = 3.0) -> np.ndarray:
    """截面 MAD 去极值后标准化。"""
    x = np.asarray(x, dtype=np.float64)
    out = np.full(x.shape, np.nan, dtype=np.float64)
    valid = np.isfinite(x)
    if valid.sum() < 3:
        return out

    xv = x[valid]
    med = np.nanmedian(xv)
    mad = np.nanmedian(np.abs(xv - med))
    if np.isfinite(mad) and mad > 1e-12:
        scale = 1.4826 * mad
        lo, hi = med - mad_n * scale, med + mad_n * scale
    else:
        lo, hi = np.nanpercentile(xv, [1.0, 99.0])

    xv = np.clip(xv, lo, hi)
    std = xv.std(ddof=0)
    if np.isfinite(std) and std > 1e-12:
        out[valid] = (xv - xv.mean()) / std
    return out


def industry_dummies(industry: pd.Series) -> np.ndarray:
    """行业哑变量，drop_first=True。"""
    codes = pd.Categorical(industry.astype(str)).codes
    n = len(codes)
    k = int(codes.max()) + 1
    if k <= 1:
        return np.empty((n, 0), dtype=np.float64)
    mat = np.zeros((n, k - 1), dtype=np.float64)
    rows = np.arange(n)
    mask = codes > 0
    mat[rows[mask], codes[mask] - 1] = 1.0
    return mat


def neutralize_by_size_industry(g: pd.DataFrame) -> pd.DataFrame:
    """单个截面内完成：原始因子去极值标准化 -> 市值行业中性化 -> 残差去极值标准化。"""
    g = g.copy().sort_values("instrument", kind="mergesort").reset_index(drop=True)
    if len(g) < MIN_CROSS_SECTION_SIZE:
        return pd.DataFrame()

    raw = g[RAW_FACTOR_NAME].to_numpy(dtype=np.float64, copy=False)
    factor_z = robust_zscore_np(raw, WINSOR_MAD_N)
    log_cap = np.log(g["float_market_cap"].to_numpy(dtype=np.float64, copy=False).clip(min=1.0))
    dummies = industry_dummies(g["industry_level1"].fillna("未知"))
    X = np.column_stack([np.ones(len(g), dtype=np.float64), log_cap, dummies])

    valid = np.isfinite(factor_z) & np.isfinite(X).all(axis=1)
    if valid.sum() < max(MIN_CROSS_SECTION_SIZE, X.shape[1] + 5):
        return pd.DataFrame()

    resid = np.full(len(g), np.nan, dtype=np.float64)
    beta = np.linalg.lstsq(X[valid], factor_z[valid], rcond=None)[0]
    resid[valid] = factor_z[valid] - X[valid] @ beta
    neutral_z = robust_zscore_np(resid, WINSOR_MAD_N)

    out = g[["date", "instrument", "float_market_cap"]].copy()
    out[NEUTRAL_FACTOR_NAME] = neutral_z.astype(np.float32)
    out = out.dropna(subset=[NEUTRAL_FACTOR_NAME, "float_market_cap"])
    return out


def calc_select_count(n: int, pct: float) -> int:
    if SELECT_COUNT_METHOD == "ceil":
        return max(1, int(np.ceil(n * pct)))
    return max(1, int(np.floor(n * pct)))


def get_current_date_from_engine(context, data) -> Optional[str]:
    if data is not None and hasattr(data, "current_dt"):
        try:
            return pd.to_datetime(data.current_dt).strftime("%Y-%m-%d")
        except Exception:
            pass
    for attr in ["current_dt", "now", "current_date"]:
        if hasattr(context, attr):
            try:
                v = getattr(context, attr)
                if v is not None:
                    return pd.to_datetime(v).strftime("%Y-%m-%d")
            except Exception:
                pass
    return None


def get_positions_dict(context) -> Dict:
    for method in ["get_positions", "get_account_positions"]:
        if hasattr(context, method):
            try:
                pos = getattr(context, method)()
                if pos is not None:
                    return pos
            except Exception:
                pass
    try:
        return context.portfolio.positions
    except Exception:
        return {}


def position_amount(pos_obj) -> float:
    for attr in ["amount", "quantity", "volume", "position"]:
        try:
            return float(getattr(pos_obj, attr))
        except Exception:
            pass
    try:
        return float(pos_obj.get("amount", 0))
    except Exception:
        return 0.0


def position_market_value(pos_obj) -> float:
    for attr in ["market_value", "value"]:
        try:
            return float(getattr(pos_obj, attr))
        except Exception:
            pass
    try:
        return float(pos_obj.get("market_value", 0))
    except Exception:
        return 0.0


def portfolio_value(context) -> float:
    for attr in ["portfolio_value", "total_value", "market_value"]:
        try:
            v = float(getattr(context.portfolio, attr))
            if np.isfinite(v) and v > 0:
                return v
        except Exception:
            pass
    return np.nan


def order_to_target_percent(context, instrument: str, weight: float) -> bool:
    for method in ["order_target_percent", "order_percent"]:
        if hasattr(context, method):
            try:
                getattr(context, method)(instrument, float(weight))
                return True
            except Exception:
                continue
    print(f"下单失败：找不到可用的目标仓位下单函数，{instrument}, target={weight:.6f}", flush=True)
    return False


# =========================
# 3. 交易日、信号日、执行日
# =========================

progress("开始获取交易日与调仓日期")
trade_dates_sql = f"""
SELECT DISTINCT date
FROM cn_stock_factors_base
WHERE date >= DATE '{START_DATE}'
  AND date <= DATE '{END_DATE}'
ORDER BY date
"""
trade_dates_df = query_df(trade_dates_sql, filters={"date": [START_DATE, END_DATE]})
trade_dates = pd.to_datetime(trade_dates_df["date"]).drop_duplicates().sort_values().reset_index(drop=True)
if len(trade_dates) < REBALANCE_DAYS + 2:
    raise ValueError("指定时间段内交易日过少，无法完成回测。")

signal_dates = trade_dates.iloc[::REBALANCE_DAYS].tolist()
signal_to_execution = {}
for dt in signal_dates:
    idx_arr = trade_dates[trade_dates == dt].index
    if len(idx_arr) == 0:
        continue
    next_idx = int(idx_arr[0]) + 1
    if next_idx < len(trade_dates):
        signal_to_execution[pd.to_datetime(dt).strftime("%Y-%m-%d")] = pd.to_datetime(trade_dates.iloc[next_idx]).strftime("%Y-%m-%d")

if not signal_to_execution:
    raise ValueError("没有可用的信号日/执行日映射。")

signal_dates = [pd.to_datetime(x) for x in signal_to_execution.keys()]
execution_dates = [pd.to_datetime(x) for x in signal_to_execution.values()]
signal_date_sql = date_in_sql(signal_dates)
execution_date_sql = date_in_sql(execution_dates)

progress(f"交易日数量：{len(trade_dates):,}；信号截面数量：{len(signal_dates):,}；调仓周期：{REBALANCE_DAYS} 个交易日")
progress(f"参与交易市值组：{SIZE_GROUPS_TO_TRADE}；每组选择因子最高前 {TOP_PCT:.2%}")


# =========================
# 4. 读取信号截面数据
# =========================

progress("开始读取信号截面数据")

signal_sql = f"""
SELECT
    b.date,
    b.instrument,
    b.float_market_cap,
    b.{INDUSTRY_COL} AS industry_level1,
    f.{FACTOR_SOURCE_COL} AS {RAW_FACTOR_NAME}
FROM cn_stock_factors_base AS b
JOIN {FACTOR_TABLE} AS f
  ON b.date = f.date AND b.instrument = f.instrument
WHERE b.date IN ({signal_date_sql})
  AND b.list_sector != 4
  AND b.st_status = 0
  AND b.suspended = 0
  AND b.float_market_cap > 0
  AND f.{FACTOR_SOURCE_COL} IS NOT NULL
ORDER BY b.date, b.instrument
"""

signal_panel = query_df(signal_sql, filters={"date": [START_DATE, END_DATE]})
if signal_panel.empty:
    raise ValueError("信号截面数据为空，请检查日期、字段名或数据权限。")

signal_panel["date"] = pd.to_datetime(signal_panel["date"]).dt.normalize()
signal_panel["instrument"] = signal_panel["instrument"].astype(str)
signal_panel["industry_level1"] = signal_panel["industry_level1"].fillna("未知").astype(str)
signal_panel = downcast_float(signal_panel, ["float_market_cap", RAW_FACTOR_NAME])
signal_panel = signal_panel.dropna(subset=["date", "instrument", "float_market_cap", RAW_FACTOR_NAME])
signal_panel = signal_panel[(signal_panel["float_market_cap"] > 0) & np.isfinite(signal_panel[RAW_FACTOR_NAME])]
signal_panel = signal_panel.drop_duplicates(subset=["date", "instrument"], keep="last")
progress(f"信号截面数据：{len(signal_panel):,} 行")


# =========================
# 5. 市值行业中性化、市值15组、组内Top 10%选股
# =========================

progress("开始逐截面中性化、市值分层与选股")
selected_parts = []
for i, (dt, g) in enumerate(signal_panel.groupby("date", sort=True), 1):
    if i == 1 or i % 10 == 0 or i == signal_panel["date"].nunique():
        progress(f"处理截面 {i}/{signal_panel['date'].nunique()}：{pd.to_datetime(dt).strftime('%Y-%m-%d')}，样本 {len(g):,}")

    neu = neutralize_by_size_industry(g)
    if neu.empty:
        continue

    neu = neu.sort_values(["float_market_cap", "instrument"], ascending=[True, True], kind="mergesort").reset_index(drop=True)
    rank = neu["float_market_cap"].rank(method="first", ascending=True)
    try:
        neu["size_group"] = pd.qcut(rank, q=N_SIZE_GROUPS, labels=list(range(1, N_SIZE_GROUPS + 1))).astype(int)
    except Exception:
        continue

    group_selected = []
    for sg_id in SIZE_GROUPS_TO_TRADE:
        sg = neu[neu["size_group"] == sg_id].copy()
        if len(sg) < MIN_STOCKS_PER_SIZE_GROUP:
            continue
        n_select = calc_select_count(len(sg), TOP_PCT)
        sg = sg.sort_values([NEUTRAL_FACTOR_NAME, "instrument"], ascending=[False, True], kind="mergesort")
        group_selected.append(sg.head(n_select))

    if group_selected:
        selected_parts.append(pd.concat(group_selected, ignore_index=True))

if not selected_parts:
    raise ValueError("没有形成任何有效选股结果，请检查市值组、样本数量或因子数据。")

selected_df = pd.concat(selected_parts, ignore_index=True)
del selected_parts, signal_panel
gc.collect()

selected_df["signal_date"] = selected_df["date"].dt.strftime("%Y-%m-%d")
selected_df["execution_date"] = selected_df["signal_date"].map(signal_to_execution)
selected_df = selected_df.dropna(subset=["execution_date"]).copy()
selected_df["stock_count"] = selected_df.groupby("signal_date")["instrument"].transform("count")
selected_df = selected_df[selected_df["stock_count"] > 0].copy()
selected_df["target_weight"] = 1.0 / selected_df["stock_count"]

signal_df = selected_df[["signal_date", "execution_date", "instrument", "size_group", NEUTRAL_FACTOR_NAME, "target_weight"]].copy()
signal_df = signal_df.sort_values(
    ["execution_date", "size_group", NEUTRAL_FACTOR_NAME, "instrument"],
    ascending=[True, True, False, True],
    kind="mergesort",
).reset_index(drop=True)

progress(f"最终信号：{len(signal_df):,} 行；涉及股票 {signal_df['instrument'].nunique():,} 只")

signal_summary = (
    signal_df.groupby(["signal_date", "execution_date"], sort=True)
    .agg(stock_count=("instrument", "count"), avg_weight=("target_weight", "mean"))
    .reset_index()
)
progress("交易信号摘要前20行：")
display(signal_summary.head(20))


# =========================
# 6. 读取执行日交易约束：开盘涨跌停、停牌
# =========================

progress("开始读取执行日交易约束")
trade_status_sql = f"""
SELECT
    date,
    instrument,
    open,
    upper_limit,
    lower_limit,
    suspended,
    st_status
FROM cn_stock_factors_base
WHERE date IN ({execution_date_sql})
ORDER BY date, instrument
"""
trade_status = query_df(trade_status_sql, filters={"date": [START_DATE, END_DATE]})
if trade_status.empty:
    raise ValueError("执行日交易约束数据为空，请检查 cn_stock_factors_base 字段或日期。")

trade_status["date"] = pd.to_datetime(trade_status["date"]).dt.strftime("%Y-%m-%d")
trade_status["instrument"] = trade_status["instrument"].astype(str)
trade_status = downcast_float(trade_status, ["open", "upper_limit", "lower_limit"])
trade_status["suspended"] = pd.to_numeric(trade_status["suspended"], errors="coerce").fillna(1).astype(int)
trade_status["st_status"] = pd.to_numeric(trade_status["st_status"], errors="coerce").fillna(0).astype(int)
trade_status = trade_status.drop_duplicates(subset=["date", "instrument"], keep="last")

valid_price = (
    np.isfinite(trade_status["open"]) &
    np.isfinite(trade_status["upper_limit"]) &
    np.isfinite(trade_status["lower_limit"]) &
    (trade_status["open"] > 0) &
    (trade_status["upper_limit"] > 0) &
    (trade_status["lower_limit"] > 0)
)
trade_status["open_limit_up"] = valid_price & (trade_status["open"] >= trade_status["upper_limit"] * (1.0 - LIMIT_EPS))
trade_status["open_limit_down"] = valid_price & (trade_status["open"] <= trade_status["lower_limit"] * (1.0 + LIMIT_EPS))
trade_status["can_buy_open"] = (trade_status["suspended"] == 0) & (~trade_status["open_limit_up"])
trade_status["can_sell_open"] = (trade_status["suspended"] == 0) & (~trade_status["open_limit_down"])

trade_status_by_date = {
    d: g.set_index("instrument")[["can_buy_open", "can_sell_open", "suspended", "open_limit_up", "open_limit_down"]].to_dict("index")
    for d, g in trade_status.groupby("date", sort=False)
}

del trade_status
gc.collect()


# =========================
# 7. BigTrader 原生回测
# =========================

progress("开始准备 BigTrader 回测输入")
backtest_data = signal_df[["execution_date", "instrument", "target_weight"]].copy()
backtest_data = backtest_data.rename(columns={"execution_date": "date"})
backtest_data["date"] = pd.to_datetime(backtest_data["date"]).dt.strftime("%Y-%m-%d")
backtest_data["instrument"] = backtest_data["instrument"].astype(str)

signal_by_execution_date = {
    d: g[["instrument", "target_weight"]].copy()
    for d, g in backtest_data.groupby("date", sort=True)
}

target_by_execution_date = {
    d: set(g["instrument"].astype(str))
    for d, g in backtest_data.groupby("date", sort=True)
}

progress("开始运行 BigTrader 原生回测")


def initialize(context):
    try:
        context.set_commission(
            bigtrader.PerOrder(
                buy_cost=BUY_COST,
                sell_cost=SELL_COST,
                min_cost=MIN_COMMISSION,
            )
        )
    except Exception as e:
        print(f"设置手续费失败，将使用引擎默认费率。原因：{e}", flush=True)

    context.signal_by_execution_date = signal_by_execution_date
    context.target_by_execution_date = target_by_execution_date
    context.trade_status_by_date = trade_status_by_date
    context.rebalance_dates = set(signal_by_execution_date.keys())

    try:
        context.subscribe_bar(list(backtest_data["instrument"].drop_duplicates()), "1d", None)
    except Exception:
        pass


def _get_trade_flags(context, current_date: str, instrument: str) -> Tuple[bool, bool]:
    row = context.trade_status_by_date.get(current_date, {}).get(str(instrument))
    if row is None:
        return False, False
    return bool(row.get("can_buy_open", False)), bool(row.get("can_sell_open", False))


def handle_data(context, data):
    current_date = get_current_date_from_engine(context, data)
    if current_date is None or current_date not in context.rebalance_dates:
        return

    today_signal = context.signal_by_execution_date.get(current_date)
    if today_signal is None or len(today_signal) == 0:
        return

    target_weights = dict(zip(today_signal["instrument"].astype(str), today_signal["target_weight"].astype(float)))
    target_instruments = set(target_weights.keys())
    positions = get_positions_dict(context)

    holding_instruments = set()
    current_weights = {}
    pv = portfolio_value(context)
    for ins, pos in positions.items():
        ins = str(ins)
        amt = position_amount(pos)
        if amt <= 0:
            continue
        holding_instruments.add(ins)
        mv = position_market_value(pos)
        if np.isfinite(pv) and pv > 0 and np.isfinite(mv):
            current_weights[ins] = mv / pv

    # 先卖出不在目标池中的股票；开盘跌停或停牌则不卖。
    for ins in sorted(holding_instruments - target_instruments):
        _, can_sell = _get_trade_flags(context, current_date, ins)
        if can_sell:
            order_to_target_percent(context, ins, 0.0)

    # 再调整目标股票。买入方向要求非开盘涨停；卖出方向要求非开盘跌停。
    for ins in sorted(target_weights.keys()):
        target_w = float(target_weights[ins])
        can_buy, can_sell = _get_trade_flags(context, current_date, ins)
        cur_w = current_weights.get(ins, 0.0)

        if cur_w <= 1e-8:
            if can_buy:
                order_to_target_percent(context, ins, target_w)
        elif target_w > cur_w + 1e-5:
            if can_buy:
                order_to_target_percent(context, ins, target_w)
        elif target_w < cur_w - 1e-5:
            if can_sell:
                order_to_target_percent(context, ins, target_w)
        else:
            # 已接近目标权重，不下单。
            pass


run_kwargs = dict(
    data=backtest_data,
    start_date=min(signal_by_execution_date.keys()),
    end_date=END_DATE,
    initialize=initialize,
    handle_data=handle_data,
    capital_base=CAPITAL_BASE,
    benchmark=BENCHMARK,
)

try:
    run_kwargs["market"] = bigtrader.Market.CN_STOCK
except Exception:
    pass

try:
    run_kwargs["frequency"] = bigtrader.Frequency.DAILY
except Exception:
    run_kwargs["frequency"] = "1d"

performance = bigtrader.run(**run_kwargs)

progress("BigTrader 回测完成")
try:
    display(performance.summary)
except Exception:
    display(performance)


从市值分层回测结果来看，该因子对于超大市值的股票能提供良好的收益率和夏普比率，这和目前所测试的其他类型的因子是不同的，其他的因子更多的是在小市值的股票上表现好，但一旦到了大市值股票当中就会失去效用，因此这个因子是对于其他类型因子的一个很好的补偿，而且财务质量因子会和其他类型的因子有较低的相关性，这对于后期做多因子组合时是有帮助的

## 市值行业中性化+防御+收益补偿市值分组回测

In [ ]:
# -*- coding: utf-8 -*-
"""
BigQuant 策略回测：qfa_roe_neutral 市值分层选股策略 + 沪深300 MA60 防御性收益补偿

因子口径：
1. 研报 qfa_roe = 单季度 ROE（平均），BigQuant 中优先使用 cn_stock_prefactors.roe_avg_mrq。
2. qfa_roe 为正向因子，原始因子直接使用 roe_avg_mrq，不取相反数。
3. 每个信号截面：原始因子先做 MAD 去极值 + 标准化，再对 log(流通市值) 与行业做截面中性化，
   最后对中性化残差再次做 MAD 去极值 + 标准化，得到 qfa_roe_neutral。

策略逻辑：
1. 每 REBALANCE_DAYS 个交易日生成一次信号，信号使用 signal_date 当日已经可得的数据。
2. 全市场按流通市值从小到大划分 15 组，1=最小市值组，15=最大市值组。
3. 通过 SIZE_GROUPS_TO_TRADE 指定参与交易的市值组，例如 [1, 2, 3]。
4. 在每个指定市值组内，选取 qfa_roe_neutral 最高的前 TOP_PCT 股票。
5. 所有入选股票等权配置。
6. 信号日后一交易日执行调仓；不在选股阶段读取执行日行情，避免未来函数。
7. 执行日使用当日开盘涨跌停状态做交易约束：开盘涨停不买入，开盘跌停不卖出；停牌不交易。
8. 股票池剔除 ST、*ST、停牌、北交所；考虑交易成本；使用 BigTrader 原生回测引擎。
9. 防御性+收益补偿：沪深300跌破 MA60 时，因子股票仓位降至10%，释放的88%仓位等权买入工商银行、交通银行、中国银行；沪深300重新站上 MA60 时，因子股票仓位恢复到98%。

注意：
- 执行日的开盘涨跌停限制只在 handle_data 当日下单时使用，不参与因子计算与选股。
- 如果某只股票因涨停无法买入或因跌停无法卖出，不做强制再平衡，避免使用执行日信息重新分配权重。
- 趋势仓位每日检查；非因子调仓日若趋势状态变化，只调整总仓位与银行补偿仓位，沿用上一期因子目标股票池。
- 为避免未来函数，t 日仓位状态默认使用 t-1 交易日已经可知的沪深300收盘价与 MA60。
"""

import gc
import time
import warnings
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

try:
    import dai
except Exception as e:
    raise ImportError("请在 BigQuant Notebook 环境中运行，本代码依赖 dai。") from e

try:
    from bigquant import bigtrader
except Exception:
    try:
        import bigtrader
    except Exception as e:
        raise ImportError("请在 BigQuant Notebook 环境中运行，本代码依赖 BigTrader。") from e

warnings.filterwarnings("ignore")


# =========================
# 1. 参数设置
# =========================

START_DATE = "2020-01-01"
END_DATE = "2026-06-30"

FACTOR_TABLE = "cn_stock_prefactors"
FACTOR_SOURCE_COL = "roe_avg_mrq"
RAW_FACTOR_NAME = "qfa_roe_raw"
NEUTRAL_FACTOR_NAME = "qfa_roe_neutral"

# 1=最小市值组，15=最大市值组
SIZE_GROUPS_TO_TRADE = [15]
N_SIZE_GROUPS = 15
TOP_PCT = 0.05
REBALANCE_DAYS = 30
MIN_STOCKS_PER_SIZE_GROUP = 20
SELECT_COUNT_METHOD = "floor"  # floor 或 ceil；至少选 1 只

# 中性化与标准化
WINSOR_MAD_N = 3.0
MIN_CROSS_SECTION_SIZE = 100
INDUSTRY_COL = "sw2021_level1"  # 可改 sw2014_level1 / cs_level1

# 回测参数
CAPITAL_BASE = 1_000_000
BENCHMARK = "000300.SH"  # 如账号环境使用 CSI 代码，可改为 000300.SH / 000300.SH / 932000.CSI 等
BUY_COST = 0.0003
SELL_COST = 0.0013
MIN_COMMISSION = 5

# 因子字段说明：qfa_roe 对应 BigQuant roe_avg_mrq，即净资产收益率(平均)(单季度)。
# 如当前账号字段权限异常，可先在 BigQuant 数据平台确认字段，或将 FACTOR_TABLE / FACTOR_SOURCE_COL 改为可用表字段。

# 执行日涨跌停判断容忍误差
LIMIT_EPS = 1e-4

# =========================
# 防御性 + 收益补偿参数
# =========================

USE_DEFENSIVE_COMPENSATION = True
TREND_INDEX_CODE = "000300.SH"      # 沪深300
TREND_MA_WINDOW = 60

# 为避免未来函数，t 日仓位调整使用 t-1 交易日收盘后已经可知的指数趋势状态。
# 如果你的 BigTrader 环境明确是“收盘后计算、下一根K线成交”，可以改成 False。
TREND_USE_PREVIOUS_TRADING_DAY = True

RISK_ON_STOCK_EXPOSURE = 0.98        # 沪深300在 MA60 上方：因子股票总仓位98%
RISK_OFF_STOCK_EXPOSURE = 0.10       # 沪深300跌破 MA60：因子股票总仓位10%
RISK_ON_DEFENSIVE_EXPOSURE = 0.00
RISK_OFF_DEFENSIVE_EXPOSURE = RISK_ON_STOCK_EXPOSURE - RISK_OFF_STOCK_EXPOSURE

DEFENSIVE_BANK_ASSETS = {
    "601398.SH": "工商银行",
    "601328.SH": "交通银行",
    "601988.SH": "中国银行",
}

START_DATE = pd.to_datetime(START_DATE).strftime("%Y-%m-%d")
END_DATE = pd.to_datetime(END_DATE).strftime("%Y-%m-%d")
QUERY_START_DATE = (pd.to_datetime(START_DATE) - pd.Timedelta(days=max(180, TREND_MA_WINDOW * 4))).strftime("%Y-%m-%d")

if not SIZE_GROUPS_TO_TRADE:
    raise ValueError("SIZE_GROUPS_TO_TRADE 不能为空。")
SIZE_GROUPS_TO_TRADE = sorted(set(int(x) for x in SIZE_GROUPS_TO_TRADE))
bad_groups = [g for g in SIZE_GROUPS_TO_TRADE if g < 1 or g > N_SIZE_GROUPS]
if bad_groups:
    raise ValueError(f"SIZE_GROUPS_TO_TRADE 中存在非法市值组：{bad_groups}，有效范围为 1~{N_SIZE_GROUPS}。")
if SELECT_COUNT_METHOD not in {"floor", "ceil"}:
    raise ValueError("SELECT_COUNT_METHOD 只能是 'floor' 或 'ceil'。")


# =========================
# 2. 通用工具函数
# =========================

_T0 = time.time()


def _elapsed() -> str:
    sec = int(time.time() - _T0)
    return f"{sec // 60:02d}:{sec % 60:02d}"


def progress(msg: str) -> None:
    print(f"[{_elapsed()}] {msg}", flush=True)


def query_df(sql: str, filters: Optional[dict] = None) -> pd.DataFrame:
    if filters is None:
        return dai.query(sql).df()
    return dai.query(sql, filters=filters).df()




def test_data_sources() -> pd.DataFrame:
    """检查当前账号是否能读取本策略需要的核心字段。"""
    tests = [
        ("cn_stock_prefactors", "roe_avg_mrq", "qfa_roe 对应字段：净资产收益率(平均)(单季度)"),
        ("cn_stock_factors_base", "float_market_cap", "流通市值"),
        ("cn_stock_factors_base", INDUSTRY_COL, "行业字段"),
        ("cn_stock_factors_base", "st_status", "ST 状态"),
        ("cn_stock_factors_base", "suspended", "停牌状态"),
        ("cn_stock_factors_base", "open", "执行日开盘价"),
        ("cn_stock_factors_base", "upper_limit", "执行日涨停价"),
        ("cn_stock_factors_base", "lower_limit", "执行日跌停价"),
        ("cn_stock_index_bar1d", "close", "沪深300收盘价候选表，用于MA60防御仓位"),
    ]
    rows = []
    for table, field, desc in tests:
        sql = f"""
        SELECT date, instrument, {field} AS value
        FROM {table}
        WHERE date >= DATE '{START_DATE}'
          AND date <= DATE '{END_DATE}'
          AND {field} IS NOT NULL
        LIMIT 5
        """
        try:
            tmp = query_df(sql, filters={"date": [START_DATE, END_DATE]})
            rows.append({
                "table": table,
                "field": field,
                "desc": desc,
                "available": not tmp.empty,
                "rows": len(tmp),
                "error": "" if not tmp.empty else "查询成功但无非空样本",
            })
        except Exception as e:
            rows.append({
                "table": table,
                "field": field,
                "desc": desc,
                "available": False,
                "rows": 0,
                "error": str(e).split("\n")[-1][:240],
            })
    out = pd.DataFrame(rows)
    display(out)
    return out

def date_in_sql(dates) -> str:
    return ", ".join(f"'{pd.to_datetime(x).strftime('%Y-%m-%d')}'" for x in dates)


def instrument_in_sql(instruments: List[str]) -> str:
    return ", ".join(f"'{str(x)}'" for x in instruments)


def downcast_float(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce", downcast="float")
    return df


def robust_zscore_np(x: np.ndarray, mad_n: float = 3.0) -> np.ndarray:
    """截面 MAD 去极值后标准化。"""
    x = np.asarray(x, dtype=np.float64)
    out = np.full(x.shape, np.nan, dtype=np.float64)
    valid = np.isfinite(x)
    if valid.sum() < 3:
        return out

    xv = x[valid]
    med = np.nanmedian(xv)
    mad = np.nanmedian(np.abs(xv - med))
    if np.isfinite(mad) and mad > 1e-12:
        scale = 1.4826 * mad
        lo, hi = med - mad_n * scale, med + mad_n * scale
    else:
        lo, hi = np.nanpercentile(xv, [1.0, 99.0])

    xv = np.clip(xv, lo, hi)
    std = xv.std(ddof=0)
    if np.isfinite(std) and std > 1e-12:
        out[valid] = (xv - xv.mean()) / std
    return out


def industry_dummies(industry: pd.Series) -> np.ndarray:
    """行业哑变量，drop_first=True。"""
    codes = pd.Categorical(industry.astype(str)).codes
    n = len(codes)
    k = int(codes.max()) + 1
    if k <= 1:
        return np.empty((n, 0), dtype=np.float64)
    mat = np.zeros((n, k - 1), dtype=np.float64)
    rows = np.arange(n)
    mask = codes > 0
    mat[rows[mask], codes[mask] - 1] = 1.0
    return mat


def neutralize_by_size_industry(g: pd.DataFrame) -> pd.DataFrame:
    """单个截面内完成：原始因子去极值标准化 -> 市值行业中性化 -> 残差去极值标准化。"""
    g = g.copy().sort_values("instrument", kind="mergesort").reset_index(drop=True)
    if len(g) < MIN_CROSS_SECTION_SIZE:
        return pd.DataFrame()

    raw = g[RAW_FACTOR_NAME].to_numpy(dtype=np.float64, copy=False)
    factor_z = robust_zscore_np(raw, WINSOR_MAD_N)
    log_cap = np.log(g["float_market_cap"].to_numpy(dtype=np.float64, copy=False).clip(min=1.0))
    dummies = industry_dummies(g["industry_level1"].fillna("未知"))
    X = np.column_stack([np.ones(len(g), dtype=np.float64), log_cap, dummies])

    valid = np.isfinite(factor_z) & np.isfinite(X).all(axis=1)
    if valid.sum() < max(MIN_CROSS_SECTION_SIZE, X.shape[1] + 5):
        return pd.DataFrame()

    resid = np.full(len(g), np.nan, dtype=np.float64)
    beta = np.linalg.lstsq(X[valid], factor_z[valid], rcond=None)[0]
    resid[valid] = factor_z[valid] - X[valid] @ beta
    neutral_z = robust_zscore_np(resid, WINSOR_MAD_N)

    out = g[["date", "instrument", "float_market_cap"]].copy()
    out[NEUTRAL_FACTOR_NAME] = neutral_z.astype(np.float32)
    out = out.dropna(subset=[NEUTRAL_FACTOR_NAME, "float_market_cap"])
    return out


def calc_select_count(n: int, pct: float) -> int:
    if SELECT_COUNT_METHOD == "ceil":
        return max(1, int(np.ceil(n * pct)))
    return max(1, int(np.floor(n * pct)))


def get_current_date_from_engine(context, data) -> Optional[str]:
    if data is not None and hasattr(data, "current_dt"):
        try:
            return pd.to_datetime(data.current_dt).strftime("%Y-%m-%d")
        except Exception:
            pass
    for attr in ["current_dt", "now", "current_date"]:
        if hasattr(context, attr):
            try:
                v = getattr(context, attr)
                if v is not None:
                    return pd.to_datetime(v).strftime("%Y-%m-%d")
            except Exception:
                pass
    return None


def get_positions_dict(context) -> Dict:
    for method in ["get_positions", "get_account_positions"]:
        if hasattr(context, method):
            try:
                pos = getattr(context, method)()
                if pos is not None:
                    return pos
            except Exception:
                pass
    try:
        return context.portfolio.positions
    except Exception:
        return {}


def position_amount(pos_obj) -> float:
    for attr in ["amount", "quantity", "volume", "position"]:
        try:
            return float(getattr(pos_obj, attr))
        except Exception:
            pass
    try:
        return float(pos_obj.get("amount", 0))
    except Exception:
        return 0.0


def position_market_value(pos_obj) -> float:
    for attr in ["market_value", "value"]:
        try:
            return float(getattr(pos_obj, attr))
        except Exception:
            pass
    try:
        return float(pos_obj.get("market_value", 0))
    except Exception:
        return 0.0


def portfolio_value(context) -> float:
    for attr in ["portfolio_value", "total_value", "market_value"]:
        try:
            v = float(getattr(context.portfolio, attr))
            if np.isfinite(v) and v > 0:
                return v
        except Exception:
            pass
    return np.nan


def order_to_target_percent(context, instrument: str, weight: float) -> bool:
    for method in ["order_target_percent", "order_percent"]:
        if hasattr(context, method):
            try:
                getattr(context, method)(instrument, float(weight))
                return True
            except Exception:
                continue
    print(f"下单失败：找不到可用的目标仓位下单函数，{instrument}, target={weight:.6f}", flush=True)
    return False


def to_date_str(x) -> str:
    return pd.to_datetime(x).strftime("%Y-%m-%d")


def to_bigtrader_instrument(inst: str) -> str:
    """兼容不同 BigQuant 环境中的股票代码后缀。"""
    s = str(inst)
    if s.endswith(".SZA"):
        return s[:-4] + ".SZ"
    if s.endswith(".SHA"):
        return s[:-4] + ".SH"
    if s.endswith(".BJA"):
        return s[:-4] + ".BJ"
    return s


def first_success_query(sql_list: List[str], err_msg: str) -> pd.DataFrame:
    last_error = None
    for sql in sql_list:
        try:
            df = query_df(sql)
            if df is not None and not df.empty:
                return df
        except Exception as e:
            last_error = e
            continue
    raise ValueError(f"{err_msg}。最后一次错误：{last_error}")


def build_hs300_ma60_allocation_df(trade_dates: pd.Series, query_start_date: str, end_date: str) -> pd.DataFrame:
    """
    构造每日趋势仓位序列。

    规则：
    - 沪深300 close >= MA60：因子股票仓位 98%，银行补偿仓位 0%；
    - 沪深300 close <  MA60：因子股票仓位 10%，释放的 88% 等权配置三只银行股；
    - 默认 t 日仓位使用 t-1 交易日趋势状态，避免在开盘/盘中读取 t 日收盘价。
    """
    trade_dates_idx = pd.DatetimeIndex(pd.to_datetime(trade_dates)).sort_values()
    if len(trade_dates_idx) == 0:
        raise ValueError("trade_dates 为空，无法构造趋势仓位序列。")

    if not USE_DEFENSIVE_COMPENSATION:
        return pd.DataFrame(
            {
                "stock_exposure": float(RISK_ON_STOCK_EXPOSURE),
                "defensive_exposure": float(RISK_ON_DEFENSIVE_EXPOSURE),
                "risk_on": True,
            },
            index=trade_dates_idx,
        )

    index_codes = []
    for code in [TREND_INDEX_CODE, BENCHMARK, "000300.SH", "000300.CSI", "000300.XSHG"]:
        if code and code not in index_codes:
            index_codes.append(code)

    index_sql_candidates: List[str] = []
    for code in index_codes:
        index_sql_candidates.extend([
            f"""
            SELECT date, instrument, close
            FROM cn_stock_index_bar1d
            WHERE instrument = '{code}'
              AND date >= DATE '{query_start_date}'
              AND date <= DATE '{end_date}'
            ORDER BY date
            """,
            f"""
            SELECT date, instrument, close
            FROM cn_stock_bar1d
            WHERE instrument = '{code}'
              AND date >= DATE '{query_start_date}'
              AND date <= DATE '{end_date}'
            ORDER BY date
            """,
        ])

    # 部分环境在 cn_stock_factors_base 中提供沪深300收盘价派生字段，作为兜底候选。
    index_sql_candidates.extend([
        f"""
        SELECT date, '000300.SH' AS instrument, MAX(hs300_close) AS close
        FROM cn_stock_factors_base
        WHERE date >= DATE '{query_start_date}'
          AND date <= DATE '{end_date}'
          AND hs300_close IS NOT NULL
        GROUP BY date
        ORDER BY date
        """,
        f"""
        SELECT date, '000300.SH' AS instrument, MAX(close_000300SH) AS close
        FROM cn_stock_factors_base
        WHERE date >= DATE '{query_start_date}'
          AND date <= DATE '{end_date}'
          AND close_000300SH IS NOT NULL
        GROUP BY date
        ORDER BY date
        """,
    ])

    idx = first_success_query(index_sql_candidates, f"无法读取沪深300日线收盘价，无法计算 MA{TREND_MA_WINDOW}")
    idx["date"] = pd.to_datetime(idx["date"])
    idx["close"] = pd.to_numeric(idx["close"], errors="coerce")
    idx = idx.dropna(subset=["date", "close"])
    idx = idx[idx["close"] > 0].copy()
    idx = idx.sort_values("date", kind="mergesort").drop_duplicates("date", keep="last")

    if len(idx) < TREND_MA_WINDOW:
        raise ValueError(f"沪深300指数数据不足，无法计算 {TREND_MA_WINDOW} 日均线。")

    close = idx.set_index("date")["close"].sort_index()
    ma = close.rolling(window=TREND_MA_WINDOW, min_periods=TREND_MA_WINDOW).mean()
    risk_on_raw = (close >= ma)
    risk_on_raw = risk_on_raw.where(ma.notna(), True)

    raw_alloc = pd.DataFrame(
        {
            "risk_on_raw": risk_on_raw.astype(bool),
            "stock_exposure_raw": np.where(risk_on_raw, RISK_ON_STOCK_EXPOSURE, RISK_OFF_STOCK_EXPOSURE),
            "defensive_exposure_raw": np.where(risk_on_raw, RISK_ON_DEFENSIVE_EXPOSURE, RISK_OFF_DEFENSIVE_EXPOSURE),
        },
        index=close.index,
    )
    raw_alloc = raw_alloc.reindex(trade_dates_idx).ffill()

    if TREND_USE_PREVIOUS_TRADING_DAY:
        risk_on = raw_alloc["risk_on_raw"].shift(1).fillna(True).astype(bool)
        stock_exposure = raw_alloc["stock_exposure_raw"].shift(1).fillna(float(RISK_ON_STOCK_EXPOSURE))
        defensive_exposure = raw_alloc["defensive_exposure_raw"].shift(1).fillna(float(RISK_ON_DEFENSIVE_EXPOSURE))
    else:
        risk_on = raw_alloc["risk_on_raw"].fillna(True).astype(bool)
        stock_exposure = raw_alloc["stock_exposure_raw"].fillna(float(RISK_ON_STOCK_EXPOSURE))
        defensive_exposure = raw_alloc["defensive_exposure_raw"].fillna(float(RISK_ON_DEFENSIVE_EXPOSURE))

    out = pd.DataFrame(
        {
            "stock_exposure": stock_exposure.astype(float),
            "defensive_exposure": defensive_exposure.astype(float),
            "risk_on": risk_on.astype(bool),
        },
        index=trade_dates_idx,
    )
    return out


# =========================
# 3. 交易日、信号日、执行日
# =========================

progress("开始获取交易日与调仓日期")
trade_dates_sql = f"""
SELECT DISTINCT date
FROM cn_stock_factors_base
WHERE date >= DATE '{START_DATE}'
  AND date <= DATE '{END_DATE}'
ORDER BY date
"""
trade_dates_df = query_df(trade_dates_sql, filters={"date": [START_DATE, END_DATE]})
trade_dates = pd.to_datetime(trade_dates_df["date"]).drop_duplicates().sort_values().reset_index(drop=True)
if len(trade_dates) < REBALANCE_DAYS + 2:
    raise ValueError("指定时间段内交易日过少，无法完成回测。")

signal_dates = trade_dates.iloc[::REBALANCE_DAYS].tolist()
signal_to_execution = {}
for dt in signal_dates:
    idx_arr = trade_dates[trade_dates == dt].index
    if len(idx_arr) == 0:
        continue
    next_idx = int(idx_arr[0]) + 1
    if next_idx < len(trade_dates):
        signal_to_execution[pd.to_datetime(dt).strftime("%Y-%m-%d")] = pd.to_datetime(trade_dates.iloc[next_idx]).strftime("%Y-%m-%d")

if not signal_to_execution:
    raise ValueError("没有可用的信号日/执行日映射。")

signal_dates = [pd.to_datetime(x) for x in signal_to_execution.keys()]
execution_dates = [pd.to_datetime(x) for x in signal_to_execution.values()]
signal_date_sql = date_in_sql(signal_dates)
execution_date_sql = date_in_sql(execution_dates)

progress(f"交易日数量：{len(trade_dates):,}；信号截面数量：{len(signal_dates):,}；调仓周期：{REBALANCE_DAYS} 个交易日")
progress(f"参与交易市值组：{SIZE_GROUPS_TO_TRADE}；每组选择因子最高前 {TOP_PCT:.2%}")

progress("开始构造沪深300 MA60 防御性收益补偿仓位序列")
trend_allocation_df = build_hs300_ma60_allocation_df(trade_dates, QUERY_START_DATE, END_DATE)
trend_summary = trend_allocation_df.loc[
    (trend_allocation_df.index >= pd.to_datetime(START_DATE)) &
    (trend_allocation_df.index <= pd.to_datetime(END_DATE))
].copy()
risk_on_days = int(trend_summary["risk_on"].sum())
risk_off_days = int((~trend_summary["risk_on"]).sum())
progress(
    f"趋势仓位序列完成：沪深300在MA60上方 {risk_on_days:,} 天，跌破MA60 {risk_off_days:,} 天；"
    f"风险开启因子股票仓位 {RISK_ON_STOCK_EXPOSURE:.2%}，"
    f"风险关闭因子股票仓位 {RISK_OFF_STOCK_EXPOSURE:.2%}，"
    f"风险关闭银行补偿仓位 {RISK_OFF_DEFENSIVE_EXPOSURE:.2%}"
)


# =========================
# 4. 读取信号截面数据
# =========================

progress("开始读取信号截面数据")

signal_sql = f"""
SELECT
    b.date,
    b.instrument,
    b.float_market_cap,
    b.{INDUSTRY_COL} AS industry_level1,
    f.{FACTOR_SOURCE_COL} AS {RAW_FACTOR_NAME}
FROM cn_stock_factors_base AS b
JOIN {FACTOR_TABLE} AS f
  ON b.date = f.date AND b.instrument = f.instrument
WHERE b.date IN ({signal_date_sql})
  AND b.list_sector != 4
  AND b.st_status = 0
  AND b.suspended = 0
  AND b.float_market_cap > 0
  AND f.{FACTOR_SOURCE_COL} IS NOT NULL
ORDER BY b.date, b.instrument
"""

signal_panel = query_df(signal_sql, filters={"date": [START_DATE, END_DATE]})
if signal_panel.empty:
    raise ValueError("信号截面数据为空，请检查日期、字段名或数据权限。")

signal_panel["date"] = pd.to_datetime(signal_panel["date"]).dt.normalize()
signal_panel["instrument"] = signal_panel["instrument"].astype(str)
signal_panel["industry_level1"] = signal_panel["industry_level1"].fillna("未知").astype(str)
signal_panel = downcast_float(signal_panel, ["float_market_cap", RAW_FACTOR_NAME])
signal_panel = signal_panel.dropna(subset=["date", "instrument", "float_market_cap", RAW_FACTOR_NAME])
signal_panel = signal_panel[(signal_panel["float_market_cap"] > 0) & np.isfinite(signal_panel[RAW_FACTOR_NAME])]
signal_panel = signal_panel.drop_duplicates(subset=["date", "instrument"], keep="last")
progress(f"信号截面数据：{len(signal_panel):,} 行")


# =========================
# 5. 市值行业中性化、市值15组、组内Top 10%选股
# =========================

progress("开始逐截面中性化、市值分层与选股")
selected_parts = []
for i, (dt, g) in enumerate(signal_panel.groupby("date", sort=True), 1):
    if i == 1 or i % 10 == 0 or i == signal_panel["date"].nunique():
        progress(f"处理截面 {i}/{signal_panel['date'].nunique()}：{pd.to_datetime(dt).strftime('%Y-%m-%d')}，样本 {len(g):,}")

    neu = neutralize_by_size_industry(g)
    if neu.empty:
        continue

    neu = neu.sort_values(["float_market_cap", "instrument"], ascending=[True, True], kind="mergesort").reset_index(drop=True)
    rank = neu["float_market_cap"].rank(method="first", ascending=True)
    try:
        neu["size_group"] = pd.qcut(rank, q=N_SIZE_GROUPS, labels=list(range(1, N_SIZE_GROUPS + 1))).astype(int)
    except Exception:
        continue

    group_selected = []
    for sg_id in SIZE_GROUPS_TO_TRADE:
        sg = neu[neu["size_group"] == sg_id].copy()
        if len(sg) < MIN_STOCKS_PER_SIZE_GROUP:
            continue
        n_select = calc_select_count(len(sg), TOP_PCT)
        sg = sg.sort_values([NEUTRAL_FACTOR_NAME, "instrument"], ascending=[False, True], kind="mergesort")
        group_selected.append(sg.head(n_select))

    if group_selected:
        selected_parts.append(pd.concat(group_selected, ignore_index=True))

if not selected_parts:
    raise ValueError("没有形成任何有效选股结果，请检查市值组、样本数量或因子数据。")

selected_df = pd.concat(selected_parts, ignore_index=True)
del selected_parts, signal_panel
gc.collect()

selected_df["signal_date"] = selected_df["date"].dt.strftime("%Y-%m-%d")
selected_df["execution_date"] = selected_df["signal_date"].map(signal_to_execution)
selected_df = selected_df.dropna(subset=["execution_date"]).copy()
selected_df["stock_count"] = selected_df.groupby("signal_date")["instrument"].transform("count")
selected_df = selected_df[selected_df["stock_count"] > 0].copy()
selected_df["target_weight"] = 1.0 / selected_df["stock_count"]

signal_df = selected_df[["signal_date", "execution_date", "instrument", "size_group", NEUTRAL_FACTOR_NAME, "target_weight"]].copy()
signal_df = signal_df.sort_values(
    ["execution_date", "size_group", NEUTRAL_FACTOR_NAME, "instrument"],
    ascending=[True, True, False, True],
    kind="mergesort",
).reset_index(drop=True)

progress(f"最终信号：{len(signal_df):,} 行；涉及股票 {signal_df['instrument'].nunique():,} 只")

signal_summary = (
    signal_df.groupby(["signal_date", "execution_date"], sort=True)
    .agg(stock_count=("instrument", "count"), avg_weight=("target_weight", "mean"))
    .reset_index()
)
progress("交易信号摘要前20行：")
display(signal_summary.head(20))



# =========================
# 6. 准备防御性收益补偿、每日触发数据与交易约束
# =========================

progress("开始准备防御资产与 BigTrader 每日触发数据")
DEFENSIVE_BANK_INSTRUMENTS = [to_bigtrader_instrument(x) for x in DEFENSIVE_BANK_ASSETS.keys()]

signal_df["instrument"] = signal_df["instrument"].map(to_bigtrader_instrument)
signal_by_execution_date = {
    d: g[["instrument", "target_weight"]].drop_duplicates("instrument").copy()
    for d, g in signal_df.groupby("execution_date", sort=True)
}

target_by_execution_date = {
    d: set(g["instrument"].astype(str))
    for d, g in signal_df.groupby("execution_date", sort=True)
}

all_backtest_instruments = sorted(
    set(signal_df["instrument"].dropna().astype(str).unique().tolist()) |
    set(DEFENSIVE_BANK_INSTRUMENTS)
)

backtest_dates = pd.to_datetime(trade_dates)
backtest_dates = backtest_dates[
    (backtest_dates >= pd.to_datetime(START_DATE)) &
    (backtest_dates <= pd.to_datetime(END_DATE))
]
backtest_date_strings = [to_date_str(x) for x in backtest_dates]

# 为了让 MA60 趋势变化能够每日触发，这里构造“每日 × 策略涉及标的”的轻量数据。
backtest_data = pd.MultiIndex.from_product(
    [backtest_date_strings, all_backtest_instruments],
    names=["date", "instrument"],
).to_frame(index=False)
backtest_data["target_weight"] = np.float32(0.0)

trend_allocation_for_bt = trend_allocation_df.loc[
    (trend_allocation_df.index >= pd.to_datetime(START_DATE)) &
    (trend_allocation_df.index <= pd.to_datetime(END_DATE))
].copy()

stock_exposure_by_date = {
    to_date_str(d): float(row["stock_exposure"])
    for d, row in trend_allocation_for_bt.iterrows()
}

defensive_exposure_by_date = {
    to_date_str(d): float(row["defensive_exposure"])
    for d, row in trend_allocation_for_bt.iterrows()
}

risk_on_by_date = {
    to_date_str(d): bool(row["risk_on"])
    for d, row in trend_allocation_for_bt.iterrows()
}

progress("防御资产：" + "、".join([f"{name}({to_bigtrader_instrument(code)})" for code, name in DEFENSIVE_BANK_ASSETS.items()]))
progress(f"BigTrader订阅标的数量：{len(all_backtest_instruments):,}；每日触发数据：{len(backtest_data):,} 行")

progress("开始读取每日交易约束：开盘涨跌停、停牌")
instrument_sql = instrument_in_sql(all_backtest_instruments)
trade_status_sql = f"""
SELECT
    date,
    instrument,
    open,
    upper_limit,
    lower_limit,
    suspended,
    st_status
FROM cn_stock_factors_base
WHERE date >= DATE '{START_DATE}'
  AND date <= DATE '{END_DATE}'
  AND instrument IN ({instrument_sql})
ORDER BY date, instrument
"""
trade_status = query_df(trade_status_sql, filters={"date": [START_DATE, END_DATE]})
if trade_status.empty:
    raise ValueError("每日交易约束数据为空，请检查 cn_stock_factors_base 字段、日期或证券代码格式。")

trade_status["date"] = pd.to_datetime(trade_status["date"]).dt.strftime("%Y-%m-%d")
trade_status["instrument"] = trade_status["instrument"].map(to_bigtrader_instrument)
trade_status = downcast_float(trade_status, ["open", "upper_limit", "lower_limit"])
trade_status["suspended"] = pd.to_numeric(trade_status["suspended"], errors="coerce").fillna(1).astype(int)
trade_status["st_status"] = pd.to_numeric(trade_status["st_status"], errors="coerce").fillna(0).astype(int)
trade_status = trade_status.drop_duplicates(subset=["date", "instrument"], keep="last")

valid_price = (
    np.isfinite(trade_status["open"]) &
    np.isfinite(trade_status["upper_limit"]) &
    np.isfinite(trade_status["lower_limit"]) &
    (trade_status["open"] > 0) &
    (trade_status["upper_limit"] > 0) &
    (trade_status["lower_limit"] > 0)
)
trade_status["open_limit_up"] = valid_price & (trade_status["open"] >= trade_status["upper_limit"] * (1.0 - LIMIT_EPS))
trade_status["open_limit_down"] = valid_price & (trade_status["open"] <= trade_status["lower_limit"] * (1.0 + LIMIT_EPS))
trade_status["can_buy_open"] = (trade_status["suspended"] == 0) & (~trade_status["open_limit_up"])
trade_status["can_sell_open"] = (trade_status["suspended"] == 0) & (~trade_status["open_limit_down"])

trade_status_by_date = {
    d: g.set_index("instrument")[["can_buy_open", "can_sell_open", "suspended", "open_limit_up", "open_limit_down"]].to_dict("index")
    for d, g in trade_status.groupby("date", sort=False)
}

try:
    del selected_df, trend_allocation_df, trend_allocation_for_bt, trade_status
except Exception:
    pass
gc.collect()


# =========================
# 7. BigTrader 原生回测
# =========================

progress("开始运行 BigTrader 原生回测")


def initialize(context):
    try:
        context.set_commission(
            bigtrader.PerOrder(
                buy_cost=BUY_COST,
                sell_cost=SELL_COST,
                min_cost=MIN_COMMISSION,
            )
        )
    except Exception as e:
        print(f"设置手续费失败，将使用引擎默认费率。原因：{e}", flush=True)

    context.signal_by_execution_date = signal_by_execution_date
    context.target_by_execution_date = target_by_execution_date
    context.trade_status_by_date = trade_status_by_date
    context.rebalance_dates = set(signal_by_execution_date.keys())

    context.stock_exposure_by_date = stock_exposure_by_date
    context.defensive_exposure_by_date = defensive_exposure_by_date
    context.risk_on_by_date = risk_on_by_date

    context.defensive_bank_instruments = DEFENSIVE_BANK_INSTRUMENTS
    context.defensive_bank_names = {
        to_bigtrader_instrument(code): name
        for code, name in DEFENSIVE_BANK_ASSETS.items()
    }

    context.current_factor_targets = {}
    context.current_target_date = None
    context.current_stock_exposure = None
    context.current_defensive_exposure = None

    try:
        context.subscribe_bar(all_backtest_instruments, "1d", None)
    except Exception:
        pass

    print("initialize 完成：qfa_roe_neutral 因子选股 + 沪深300 MA60 防御性收益补偿策略", flush=True)
    print("防御资产：" + "、".join([f"{context.defensive_bank_names.get(x, x)}({x})" for x in context.defensive_bank_instruments]), flush=True)


def _get_trade_flags(context, current_date: str, instrument: str) -> Tuple[bool, bool]:
    row = context.trade_status_by_date.get(current_date, {}).get(str(instrument))
    if row is None:
        return False, False
    return bool(row.get("can_buy_open", False)), bool(row.get("can_sell_open", False))


def _get_stock_exposure(context, date_str: str) -> float:
    try:
        return float(context.stock_exposure_by_date.get(date_str, RISK_ON_STOCK_EXPOSURE))
    except Exception:
        return float(RISK_ON_STOCK_EXPOSURE)


def _get_defensive_exposure(context, date_str: str) -> float:
    try:
        return float(context.defensive_exposure_by_date.get(date_str, RISK_ON_DEFENSIVE_EXPOSURE))
    except Exception:
        return float(RISK_ON_DEFENSIVE_EXPOSURE)


def _allocation_changed(context, stock_exposure: float, defensive_exposure: float) -> bool:
    old_stock = getattr(context, "current_stock_exposure", None)
    old_def = getattr(context, "current_defensive_exposure", None)
    if old_stock is None or old_def is None:
        return True
    return (
        abs(float(old_stock) - float(stock_exposure)) > 1e-8 or
        abs(float(old_def) - float(defensive_exposure)) > 1e-8
    )


def _order_to_target_with_trade_constraint(context, current_date: str, instrument: str, target_w: float, current_w: float) -> None:
    """根据目标仓位变化方向，分别检查开盘涨停/跌停/停牌约束。"""
    can_buy, can_sell = _get_trade_flags(context, current_date, instrument)
    target_w = float(target_w)
    current_w = float(current_w)

    if current_w <= 1e-8 and target_w <= 1e-8:
        return
    if target_w > current_w + 1e-5:
        if can_buy:
            order_to_target_percent(context, instrument, target_w)
    elif target_w < current_w - 1e-5:
        if can_sell:
            order_to_target_percent(context, instrument, target_w)
    else:
        return


def handle_data(context, data):
    current_date = get_current_date_from_engine(context, data)
    if current_date is None:
        return

    stock_exposure = _get_stock_exposure(context, current_date)
    defensive_exposure = _get_defensive_exposure(context, current_date)
    risk_on = bool(context.risk_on_by_date.get(current_date, True))

    is_rebalance_date = current_date in context.rebalance_dates
    allocation_changed = _allocation_changed(context, stock_exposure, defensive_exposure)

    # 非调仓日且趋势仓位没有变化时，不重复下单。
    if (not is_rebalance_date) and (not allocation_changed):
        return

    # 调仓日更新因子目标股票池；非调仓日只在趋势变化时沿用上一期目标池调整仓位。
    if is_rebalance_date:
        today_signal = context.signal_by_execution_date.get(current_date)
        if today_signal is None or len(today_signal) == 0:
            factor_targets = {}
        else:
            today_signal = today_signal.copy()
            today_signal["instrument"] = today_signal["instrument"].astype(str).map(to_bigtrader_instrument)
            today_signal["target_weight"] = pd.to_numeric(today_signal["target_weight"], errors="coerce")
            today_signal = today_signal.dropna(subset=["instrument", "target_weight"])
            # target_weight 在信号里合计为 1；这里再乘以当日趋势股票总仓位。
            factor_targets = dict(zip(today_signal["instrument"], today_signal["target_weight"]))
        context.current_factor_targets = factor_targets
        context.current_target_date = current_date
    else:
        factor_targets = dict(getattr(context, "current_factor_targets", {}))

    factor_target_weights = {
        str(ins): float(base_w) * float(stock_exposure)
        for ins, base_w in factor_targets.items()
        if np.isfinite(float(base_w)) and float(base_w) > 0
    }

    defensive_targets = list(getattr(context, "defensive_bank_instruments", []))
    defensive_weight = float(defensive_exposure) / len(defensive_targets) if defensive_targets else 0.0
    defensive_target_weights = {str(ins): defensive_weight for ins in defensive_targets}

    final_target_weights = {}
    final_target_weights.update(factor_target_weights)
    final_target_weights.update(defensive_target_weights)

    positions = get_positions_dict(context)
    holding_instruments = set()
    current_weights: Dict[str, float] = {}
    pv = portfolio_value(context)
    for ins, pos in positions.items():
        ins = to_bigtrader_instrument(ins)
        amt = position_amount(pos)
        if amt <= 0:
            continue
        holding_instruments.add(ins)
        mv = position_market_value(pos)
        if np.isfinite(pv) and pv > 0 and np.isfinite(mv):
            current_weights[ins] = mv / pv
        else:
            current_weights[ins] = 0.0

    # 先卖出不在目标集合中的股票；跌停或停牌则暂不强卖。
    target_instruments = set(final_target_weights.keys())
    for ins in sorted(holding_instruments - target_instruments):
        _order_to_target_with_trade_constraint(context, current_date, ins, 0.0, current_weights.get(ins, 0.0))

    # 再调整目标股票和银行补偿资产。
    for ins in sorted(final_target_weights.keys()):
        target_w = float(final_target_weights[ins])
        cur_w = current_weights.get(ins, 0.0)
        _order_to_target_with_trade_constraint(context, current_date, ins, target_w, cur_w)

    context.current_stock_exposure = stock_exposure
    context.current_defensive_exposure = defensive_exposure

    state_text = f"风险开启/沪深300在MA{TREND_MA_WINDOW}上方" if risk_on else f"风险关闭/沪深300跌破MA{TREND_MA_WINDOW}"
    n_factor = len(factor_target_weights)
    n_bank = len(defensive_targets)
    if is_rebalance_date:
        print(
            f"{current_date} 调仓：{state_text}，因子股票 {n_factor} 只，"
            f"因子股票总仓位 {stock_exposure:.2%}，银行补偿总仓位 {defensive_exposure:.2%}，"
            f"单只银行 {defensive_weight:.4f}",
            flush=True,
        )
    elif allocation_changed:
        print(
            f"{current_date} 趋势仓位切换：{state_text}，沿用 {context.current_target_date} 目标池，"
            f"因子股票总仓位 {stock_exposure:.2%}，银行补偿总仓位 {defensive_exposure:.2%}，"
            f"银行数量 {n_bank}",
            flush=True,
        )


run_kwargs = dict(
    data=backtest_data,
    start_date=START_DATE,
    end_date=END_DATE,
    initialize=initialize,
    handle_data=handle_data,
    capital_base=CAPITAL_BASE,
    benchmark=BENCHMARK,
)

try:
    run_kwargs["market"] = bigtrader.Market.CN_STOCK
except Exception:
    pass

try:
    run_kwargs["frequency"] = bigtrader.Frequency.DAILY
except Exception:
    run_kwargs["frequency"] = "1d"

performance = bigtrader.run(**run_kwargs)

progress("BigTrader 回测完成")
try:
    display(performance.summary)
except Exception:
    display(performance)


加入防御+收益补偿后，这个策略的提升是比较明显的，而且提升的核心并不只是累计收益从原来的约 220% 小幅提高到 228%，更重要的是风险收益结构明显优化：最大回撤从此前接近 48% 大幅降到 16.55%，收益波动率也从约 25.93% 降到 18.25%，夏普比率提升到 0.97，盈亏比提升到 2.18，说明 MA60 防御机制有效避开了沪深300下行阶段中大市值股票的系统性回撤，而降下来的仓位配置到工行、交行、中行这类低波动、高股息、偏防御银行股，又避免了简单空仓导致的收益断层；因此这套改动本质上是把原本“高收益但回撤过大”的大市值质量因子策略，改造成了一个更接近可实盘接受的稳健型策略。不过也要注意，它的胜率仍然只有 47.57%，阿尔法从此前约 0.19 降到 0.18，说明收益来源并没有发生本质增强，主要改善来自择时降风险和银行补偿降低回撤；后续最好继续测试 MA40、MA80、MA120，以及防御资产是否换成银行组合、红利低波、货币基金替代，确认这不是单一参数和单一防御资产带来的过拟合结果。